<a href="https://colab.research.google.com/github/wesleyc00/helmet_detection_yolo_and_torchvision/blob/main/TCC_HELMET_DETECTION.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## DETECÇÃO AUTOMÁTICA DO USO DE CAPACETE POR MOTOCICLISTAS UTILIZANDO REDES YOLO E VISÃO COMPUTACIONAL

## Extrair pasta

In [ ]:
# @title
import zipfile
import os

caminho_zip = '/content/todos_experimentos.zip'
pasta_destino = '/content'

with zipfile.ZipFile(caminho_zip, 'r') as zip_ref:
    zip_ref.extractall(pasta_destino)

print("extraído!")

# Treinar modelos YOLO

## Treinar YOLO v8n com 5 sementes

In [ ]:
# @title
pip install ultralytics

In [ ]:
# @title
from ultralytics import YOLO

seeds = [42, 123, 2025, 7, 21]

for seed in seeds:

    modelo = YOLO("yolov8n.pt")

    modelo.train(
        data="/content/dataset/dataset_tratado/data.yaml",
        epochs=150,
        imgsz=640,
        batch=16,
        device=0,
        workers=2,
        seed=seed,
        deterministic=True,
        patience=40,
        project="/content/experimentos",
        name=f"yolov8n_seed_{seed}",
        exist_ok=True,
        plots=True
    )

## Treinar YOLO v9t (5 seeds)

In [ ]:
# @title
from ultralytics import YOLO

seeds = [42, 123, 2025, 7, 21]

for seed in seeds:

    modelo = YOLO("yolov9t.pt")

    modelo.train(
        data="/content/dataset/dataset_tratado/data.yaml",
        epochs=150,
        imgsz=640,
        batch=16,
        device=0,
        workers=2,
        seed=seed,
        deterministic=True,
        patience=40,
        project="/content/experimentos",
        name=f"yolov9t_seed_{seed}",
        exist_ok=True,
        plots=True
    )

## Treinamento do YOLO v10n (5 seeds)

In [ ]:
# @title
from ultralytics import YOLO

seeds = [42, 123, 2025, 7, 21]

for seed in seeds:

    modelo = YOLO("yolov10n.pt")

    modelo.train(
        data="/content/dataset/dataset_tratado/data.yaml",
        epochs=150,
        imgsz=640,
        batch=16,
        device=0,
        workers=2,
        seed=seed,
        deterministic=True,
        patience=40,
        project="/content/experimentos",
        name=f"yolov10n_seed_{seed}",
        exist_ok=True,
        plots=True
    )

In [ ]:
# @title
!zip -r /content/experimentos.zip /content/experimentos

# Avaliação dos modelos YOLO

## Avaliar os 15 modelos no conjunto de teste

In [ ]:
# @title
!pip install -q ultralytics pandas

## Conferir dataset

In [ ]:
# @title

from pathlib import Path
from collections import Counter

# Caminhos do dataset corrigido
base = Path("/content/dataset_tratado")
imagens_dir = base / "test" / "images"
labels_dir = base / "test" / "labels"

# Verificar existência das pastas
assert imagens_dir.exists(), f"Pasta não encontrada: {imagens_dir}"
assert labels_dir.exists(), f"Pasta não encontrada: {labels_dir}"

# Contar imagens e anotações
extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

imagens = [
    p for p in imagens_dir.iterdir()
    if p.suffix.lower() in extensoes
]

labels = list(labels_dir.glob("*.txt"))

nomes_labels = {p.stem for p in labels}
nomes_imagens = {p.stem for p in imagens}

sem_arquivo_label = nomes_imagens - nomes_labels

# Contar objetos por classe
contador = Counter()

for arquivo in labels:
    with open(arquivo, "r") as f:
        for linha in f:
            partes = linha.strip().split()
            if len(partes) >= 5:
                contador[partes[0]] += 1

print("=" * 50)
print("VERIFICAÇÃO DO DATASET CORRIGIDO")
print("=" * 50)

print(f"Total de imagens: {len(imagens)}")
print(f"Arquivos de anotação: {len(labels)}")
print(f"Imagens sem arquivo de anotação: {len(sem_arquivo_label)}")

print("\nQuantidade de objetos por classe:")
print(f"With Helmet (0): {contador['0']}")
print(f"Without Helmet (1): {contador['1']}")

print("\nVerificação concluída.")


## Checkpoints

In [ ]:
# @title

from pathlib import Path

raiz = Path("/content/todos_experimentos")

modelos = ["yolov8n", "yolov9t", "yolov10n"]
sementes = [42, 123, 2025, 7, 21]

print("=" * 65)
print("VERIFICAÇÃO DOS CHECKPOINTS YOLO")
print("=" * 65)

for modelo in modelos:
    print(f"\nMODELO: {modelo}")

    for seed in sementes:
        checkpoint = (
            raiz / f"{modelo}_seed_{seed}" / "weights" / "best.pt"
        )

        if checkpoint.exists():
            print(f"Seed {seed}: ENCONTRADO")
        else:
            print(f"Seed {seed}: NÃO ENCONTRADO")


## Configuração

In [ ]:
# @title

from pathlib import Path
import torch
import numpy as np
import pandas as pd
import gc
from ultralytics import YOLO

# Dataset corrigido
PASTA_DATASET = Path("/content/dataset_tratado")
PASTA_TESTE = PASTA_DATASET / "test"

# Experimentos
PASTA_EXPERIMENTOS = Path("/content/todos_experimentos")

# Resultados
PASTA_AVALIACOES = Path("/content/avaliacoes_finais/yolo")
PASTA_AVALIACOES.mkdir(parents=True, exist_ok=True)

# Modelos e sementes
MODELOS = ["yolov8n", "yolov9t", "yolov10n"]
SEEDS = [42, 123, 2025, 7, 21]

# Parâmetros
IMG_SIZE = 640
DEVICE = 0
CONF_MAP = 0.001
CONF_PR = 0.25
IOU_NMS = 0.7

# Arquivos de teste
imagens_teste = sorted(
    list((PASTA_TESTE / "images").glob("*.jpg")) +
    list((PASTA_TESTE / "images").glob("*.jpeg")) +
    list((PASTA_TESTE / "images").glob("*.png"))
)

print("Configuração concluída.")
print("GPU:", torch.cuda.get_device_name(0))
print("Imagens de teste:", len(imagens_teste))
print("Modelos:", MODELOS)
print("Sementes:", SEEDS)
print("Total de avaliações:", len(MODELOS) * len(SEEDS))


## Avaliação do mAP

In [ ]:
# @title

from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import gc
import torch

resultados_map = []

for nome in MODELOS:
    for seed in SEEDS:

        print("\n" + "=" * 60)
        print(f"MODELO: {nome} | SEED: {seed}")
        print("=" * 60)

        checkpoint = (
            PASTA_EXPERIMENTOS /
            f"{nome}_seed_{seed}" /
            "weights" / "best.pt"
        )

        modelo = YOLO(str(checkpoint))

        resultado = modelo.val(
            data=str(PASTA_DATASET / "data.yaml"),
            split="test",
            imgsz=IMG_SIZE,
            batch=16,
            conf=CONF_MAP,
            iou=IOU_NMS,
            device=DEVICE,
            plots=False,
            verbose=False
        )

        resultados_map.append({
            "Modelo": nome,
            "Seed": seed,
            "mAP50": float(resultado.box.map50),
            "mAP50-95": float(resultado.box.map),
            "Precisao_conf001": float(resultado.box.mp),
            "Recall_conf001": float(resultado.box.mr)
        })

        print(f"mAP50: {resultado.box.map50:.4f}")
        print(f"mAP50-95: {resultado.box.map:.4f}")

        del modelo, resultado
        gc.collect()
        torch.cuda.empty_cache()

df_map = pd.DataFrame(resultados_map)

display(df_map.round(4))

df_map.to_csv(
    PASTA_AVALIACOES / "resultados_map_conf001.csv",
    index=False
)

print("\nAvaliação de mAP finalizada.")


## Precision, Recall e matriz de confusão com conf= 0.25

In [ ]:
# @title

resultados_pr = []

for nome in MODELOS:
    for seed in SEEDS:

        print("\n" + "=" * 60)
        print(f"MODELO: {nome} | SEED: {seed}")
        print("=" * 60)

        checkpoint = (
            PASTA_EXPERIMENTOS /
            f"{nome}_seed_{seed}" /
            "weights" / "best.pt"
        )

        modelo = YOLO(str(checkpoint))

        resultado = modelo.val(
            data=str(PASTA_DATASET / "data.yaml"),
            split="test",
            imgsz=IMG_SIZE,
            batch=16,
            conf=CONF_PR,
            iou=IOU_NMS,
            device=DEVICE,
            plots=True,
            save_json=True,
            verbose=False
        )

        resultados_pr.append({
            "Modelo": nome,
            "Seed": seed,
            "Precisao": float(resultado.box.mp),
            "Recall": float(resultado.box.mr),
            "F1": float(
                2 * resultado.box.mp * resultado.box.mr /
                (resultado.box.mp + resultado.box.mr + 1e-16)
            )
        })

        print(f"Precisão: {resultado.box.mp:.4f}")
        print(f"Recall: {resultado.box.mr:.4f}")

        # Salvar matriz de confusão
        pasta_resultado = Path(resultado.save_dir)
        print("Resultados:", pasta_resultado)

        del modelo, resultado
        gc.collect()
        torch.cuda.empty_cache()

df_pr = pd.DataFrame(resultados_pr)

display(df_pr.round(4))

df_pr.to_csv(
    PASTA_AVALIACOES / "resultados_pr_conf025.csv",
    index=False
)

print("\nAvaliação de precisão e recall finalizada.")


## Consolidar os resultados YOLO

In [ ]:
# @title

# Verificar se todas as avaliações foram realizadas
assert len(df_map) == 15, "A avaliação de mAP não possui 15 resultados."
assert len(df_pr) == 15, "A avaliação de P/R não possui 15 resultados."

# Unir os resultados
df_yolo_final = pd.merge(
    df_map[["Modelo", "Seed", "mAP50", "mAP50-95"]],
    df_pr[["Modelo", "Seed", "Precisao", "Recall", "F1"]],
    on=["Modelo", "Seed"],
    how="inner",
    validate="one_to_one"
)

# Ordenar
df_yolo_final = df_yolo_final.sort_values(
    ["Modelo", "Seed"]
).reset_index(drop=True)

# Exibir
display(df_yolo_final.round(4))

# Salvar
caminho_csv = PASTA_AVALIACOES / "resultados_yolo_consolidados.csv"

df_yolo_final.to_csv(caminho_csv, index=False)

print("\nTotal de avaliações:", len(df_yolo_final))
print("Arquivo salvo em:", caminho_csv)


## Verificar o melhor ponto de F1

In [ ]:
# @title
from pathlib import Path

RAIZ = Path("/content/todos_experimentos")

curvas = []

for modelo in ["yolov8n", "yolov9t", "yolov10n"]:
    for seed in [42, 123, 2025, 7, 21]:

        encontrados = list(
            RAIZ.rglob(
                f"{modelo}_seed_{seed}/**/BoxF1_curve.png"
            )
        )

        for arquivo in encontrados:
            curvas.append({
                "Modelo": modelo,
                "Seed": seed,
                "Arquivo": str(arquivo)
            })

df_curvas = pd.DataFrame(curvas)

display(df_curvas)

print("\nTotal de curvas encontradas:", len(df_curvas))

## Inspecionar Dados Numéricos das Curvas F1

In [ ]:
# @title
from ultralytics import YOLO
from pathlib import Path

# Carregar um modelo de referência para instanciar o objeto 'resultado'
caminho_checkpoint = Path("/content/todos_experimentos/yolov8n_seed_42/weights/best.pt")
modelo = YOLO(str(caminho_checkpoint))

# Gerar o resultado executando a validação no teste
resultado = modelo.val(
    data="/content/dataset_tratado/data.yaml",
    split="test",
    imgsz=640,
    batch=16,
    conf=0.001,
    device=0,
    plots=False,
    verbose=False
)

metricas = resultado.box

print("Formato dos arrays:\n")

for atributo in ["px", "p_curve", "r_curve", "f1_curve"]:
    valores = getattr(metricas, atributo)

    print(
        f"{atributo}: "
        f"shape={valores.shape}, "
        f"tipo={type(valores)}"
    )

print("\nNúmero de classes:", metricas.nc)

print("\nPrimeiros limiares:")
print(metricas.px[:10])

print("\nMelhor F1 por classe:")
print(metricas.f1_curve.max(axis=1))

## Calcular Melhor F1 para Todos os Modelos YOLO (Usando validação)

In [ ]:
# @title
from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np

RAIZ = Path("/content/todos_experimentos")

MODELOS = ["yolov8n", "yolov9t", "yolov10n"]
SEEDS = [42, 123, 2025, 7, 21]

resultados_f1 = []

for nome_modelo in MODELOS:
    for seed in SEEDS:

        print(f"\nAvaliando {nome_modelo} | Seed {seed}")

        checkpoint = (
            RAIZ
            / f"{nome_modelo}_seed_{seed}"
            / "weights"
            / "best.pt"
        )

        assert checkpoint.exists(), f"Checkpoint não encontrado: {checkpoint}"

        modelo = YOLO(str(checkpoint))

        resultado = modelo.val(
            data="/content/dataset/data.yaml",
            split="val",
            imgsz=640,
            batch=16,
            conf=0.001,
            device=0,
            plots=False,
            verbose=False
        )

        metricas = resultado.box

        f1_medio = metricas.f1_curve.mean(axis=0)
        indice = np.argmax(f1_medio)

        registro = {
            "Modelo": nome_modelo,
            "Seed": seed,
            "Melhor_Conf": float(metricas.px[indice]),
            "F1_Medio": float(f1_medio[indice])
        }

        for classe, nome in enumerate(
            ["With_Helmet", "Without_Helmet"]
        ):
            registro[f"Precisao_{nome}"] = float(
                metricas.p_curve[classe, indice]
            )
            registro[f"Recall_{nome}"] = float(
                metricas.r_curve[classe, indice]
            )
            registro[f"F1_{nome}"] = float(
                metricas.f1_curve[classe, indice]
            )

        resultados_f1.append(registro)

df_yolo_melhor_f1 = pd.DataFrame(resultados_f1)

display(df_yolo_melhor_f1.round(4))

arquivo = Path("/content/avaliacoes_finais/yolo")
arquivo.mkdir(parents=True, exist_ok=True)

df_yolo_melhor_f1.to_csv(
    arquivo / "resultados_melhor_f1_validacao.csv",
    index=False
)

print("\nTotal de avaliações:", len(df_yolo_melhor_f1))
print("Resultados salvos com sucesso.")

## Avaliar YOLO no Teste com Melhor F1 da Validação

In [ ]:
# @title

from ultralytics import YOLO
from pathlib import Path
import pandas as pd
import numpy as np

RAIZ = Path("/content/todos_experimentos")

SEEDS = [42, 123, 2025, 7, 21]
MODELOS = ["yolov8n", "yolov9t", "yolov10n"]

assert len(df_yolo_melhor_f1) == 15, "A tabela de limiares deve conter 15 experimentos."

resultados_teste = []

for _, linha in df_yolo_melhor_f1.iterrows():

    nome_modelo = linha["Modelo"]
    seed = int(linha["Seed"])
    melhor_conf = float(linha["Melhor_Conf"])

    print(f"\nAvaliando {nome_modelo} | Seed {seed}")

    checkpoint = (
        RAIZ
        / f"{nome_modelo}_seed_{seed}"
        / "weights"
        / "best.pt"
    )

    assert checkpoint.exists(), f"Checkpoint não encontrado: {checkpoint}"

    modelo = YOLO(str(checkpoint))

    resultado = modelo.val(
        data="/content/dataset/data.yaml",
        split="test",
        imgsz=640,
        batch=16,
        conf=0.001,
        device=0,
        plots=False,
        verbose=False
    )

    metricas = resultado.box

    # Localiza na curva de teste o limiar escolhido na validação
    indice = int(np.argmin(np.abs(metricas.px - melhor_conf)))

    registro = {
        "Modelo": nome_modelo,
        "Seed": seed,
        "Melhor_Conf_Validacao": melhor_conf,
        "Conf_Usada_Teste": float(metricas.px[indice])
    }

    for classe, nome in enumerate(["With_Helmet", "Without_Helmet"]):

        registro[f"Precisao_{nome}"] = float(
            metricas.p_curve[classe, indice]
        )

        registro[f"Recall_{nome}"] = float(
            metricas.r_curve[classe, indice]
        )

        registro[f"F1_{nome}"] = float(
            metricas.f1_curve[classe, indice]
        )

    registro["F1_Medio_Teste"] = float(
        metricas.f1_curve[:, indice].mean()
    )

    resultados_teste.append(registro)

df_yolo_pr_teste = pd.DataFrame(resultados_teste)

display(df_yolo_pr_teste.round(4))

PASTA = Path("/content/avaliacoes_finais/yolo")
PASTA.mkdir(parents=True, exist_ok=True)

df_yolo_pr_teste.to_csv(
    PASTA / "resultados_yolo_pr_teste_melhor_f1.csv",
    index=False
)

print("\nTotal de avaliações:", len(df_yolo_pr_teste))
print("Resultados salvos com sucesso.")


## Consolidar resultados finais YOLO

In [ ]:
# @title
import pandas as pd
from pathlib import Path

# Conferir se temos os 15 resultados de cada etapa
assert len(df_map) == 15, "A tabela de mAP não contém 15 experimentos."
assert len(df_yolo_pr_teste) == 15, "A tabela de teste não contém 15 experimentos."

# Selecionar as métricas de mAP
df_map_final = df_map[
    ["Modelo", "Seed", "mAP50", "mAP50-95"]
].copy()

# Selecionar as métricas de Precisão, Recall e F1 do teste
colunas_teste = [
    "Modelo",
    "Seed",
    "Melhor_Conf_Validacao",
    "Precisao_With_Helmet",
    "Recall_With_Helmet",
    "F1_With_Helmet",
    "Precisao_Without_Helmet",
    "Recall_Without_Helmet",
    "F1_Without_Helmet",
    "F1_Medio_Teste"
]

df_teste_final = df_yolo_pr_teste[colunas_teste].copy()

# Consolidar
df_yolo_final = pd.merge(
    df_map_final,
    df_teste_final,
    on=["Modelo", "Seed"],
    how="inner",
    validate="one_to_one"
)

df_yolo_final = (
    df_yolo_final
    .sort_values(["Modelo", "Seed"])
    .reset_index(drop=True)
)

assert len(df_yolo_final) == 15
assert not df_yolo_final.isnull().any().any()

display(df_yolo_final.round(4))

# Salvar CSV
PASTA = Path("/content/avaliacoes_finais/yolo")
PASTA.mkdir(parents=True, exist_ok=True)

df_yolo_final.to_csv(
    PASTA / "resultados_yolo_consolidados.csv",
    index=False
)

print("\nTotal de experimentos:", len(df_yolo_final))
print("Arquivo consolidado salvo com sucesso.")

## Estatísticas Descritivas dos Modelos YOLO

In [ ]:
# @title

import pandas as pd
import numpy as np
from scipy import stats
from pathlib import Path

METRICAS = [
    "mAP50",
    "mAP50-95",
    "Precisao_With_Helmet",
    "Recall_With_Helmet",
    "F1_With_Helmet",
    "Precisao_Without_Helmet",
    "Recall_Without_Helmet",
    "F1_Without_Helmet",
    "F1_Medio_Teste"
]

resultados_estatisticos = []

for modelo in ["yolov8n", "yolov9t", "yolov10n"]:

    dados_modelo = df_yolo_final[
        df_yolo_final["Modelo"] == modelo
    ]

    assert len(dados_modelo) == 5, (
        f"{modelo}: esperado 5 seeds, encontrado {len(dados_modelo)}"
    )

    for metrica in METRICAS:

        valores = dados_modelo[metrica].dropna().astype(float)

        n = len(valores)
        media = valores.mean()
        desvio = valores.std(ddof=1)
        erro_padrao = stats.sem(valores)

        t_critico = stats.t.ppf(0.975, df=n - 1)
        margem = t_critico * erro_padrao

        resultados_estatisticos.append({
            "Modelo": modelo,
            "Metrica": metrica,
            "N": n,
            "Media": media,
            "Mediana": valores.median(),
            "Desvio_Padrao": desvio,
            "Minimo": valores.min(),
            "Maximo": valores.max(),
            "IC95_Inferior": media - margem,
            "IC95_Superior": media + margem
        })

df_estatisticas_yolo = pd.DataFrame(resultados_estatisticos)

display(df_estatisticas_yolo.round(4))

PASTA = Path("/content/avaliacoes_finais/yolo")
PASTA.mkdir(parents=True, exist_ok=True)

df_estatisticas_yolo.to_csv(
    PASTA / "estatisticas_descritivas_yolo.csv",
    index=False
)

print("Estatísticas calculadas para os três modelos.")
print("Total de combinações:", len(df_estatisticas_yolo))


## Parâmetros, GFLOPs e Tamanho dos Modelos YOLO

In [ ]:
# @title
# instalar fvcore
!pip -q install fvcore

In [ ]:
# @title
import torch
import pandas as pd
from pathlib import Path
from ultralytics import YOLO
from fvcore.nn import FlopCountAnalysis

RAIZ = Path("/content/todos_experimentos")
PASTA = Path("/content/avaliacoes_finais/yolo")
PASTA.mkdir(parents=True, exist_ok=True)

MODELOS = ["yolov8n", "yolov9t", "yolov10n"]
SEEDS = [42, 123, 2025, 7, 21]

device = torch.device("cuda:0")
resultados_complexidade = []

for nome_modelo in MODELOS:
    for seed in SEEDS:

        print(f"\nModelo: {nome_modelo} | Seed: {seed}")

        checkpoint = (
            RAIZ / f"{nome_modelo}_seed_{seed}" / "weights" / "best.pt"
        )

        assert checkpoint.exists(), f"Checkpoint não encontrado: {checkpoint}"

        modelo = YOLO(str(checkpoint))

        # Fusão das camadas antes da análise
        rede = modelo.model.to(device).float().eval()
        rede.fuse()
        rede.eval()

        # Parâmetros
        parametros = sum(p.numel() for p in rede.parameters())

        # Tamanho do checkpoint original
        tamanho_mb = checkpoint.stat().st_size / (1024 ** 2)

        # GFLOPs com fvcore
        entrada = torch.randn(1, 3, 640, 640, device=device)

        analise = FlopCountAnalysis(rede, (entrada,))
        analise.unsupported_ops_warnings(False)
        analise.uncalled_modules_warnings(False)

        macs = analise.total()
        gflops = (macs * 2) / 1e9

        resultados_complexidade.append({
            "Modelo": nome_modelo,
            "Seed": seed,
            "Parametros": parametros,
            "Tamanho_MB": tamanho_mb,
            "MACs_G": macs / 1e9,
            "GFLOPs": gflops
        })

        print(f"Parâmetros: {parametros:,}")
        print(f"Tamanho: {tamanho_mb:.2f} MB")
        print(f"GFLOPs: {gflops:.3f}")

df_complexidade_yolo = pd.DataFrame(resultados_complexidade)

display(df_complexidade_yolo.round(3))

# Resumo por arquitetura
resumo_complexidade_yolo = (
    df_complexidade_yolo
    .groupby("Modelo")[["Parametros", "Tamanho_MB", "GFLOPs"]]
    .agg(["mean", "std"])
)

display(resumo_complexidade_yolo.round(3))

df_complexidade_yolo.to_csv(
    PASTA / "complexidade_yolo.csv",
    index=False
)

resumo_complexidade_yolo.to_csv(
    PASTA / "resumo_complexidade_yolo.csv"
)

print("\nTotal de checkpoints analisados:", len(df_complexidade_yolo))


## Medir Latência e FPS dos Modelos YOLO

In [ ]:
# @title

import torch
import time
import numpy as np
import pandas as pd
from ultralytics import YOLO
from pathlib import Path

RAIZ = Path("/content/todos_experimentos")

MODELOS = ["yolov8n", "yolov9t", "yolov10n"]
SEEDS = [42, 123, 2025, 7, 21]

assert torch.cuda.is_available(), "GPU CUDA não disponível."

device = torch.device("cuda:0")

print("GPU:", torch.cuda.get_device_name(0))
print("Precisão: FP32")
print("Resolução: 640x640")
print("Batch size: 1")
print("Fusão de camadas: ativada")
print("NMS: não incluído")

resultados_yolo = []

for nome_modelo in MODELOS:
    for seed in SEEDS:

        print(f"\nMedindo {nome_modelo} | Seed {seed}")

        checkpoint = (
            RAIZ
            / f"{nome_modelo}_seed_{seed}"
            / "weights"
            / "best.pt"
        )

        modelo = YOLO(str(checkpoint))

        # Fusão das camadas antes das medições
        rede = modelo.model.to(device).float().eval()
        rede.fuse()
        rede.eval()

        # Parâmetros após fusão
        parametros = sum(
            p.numel() for p in rede.parameters()
        )

        # Tamanho do checkpoint original
        tamanho_mb = checkpoint.stat().st_size / (1024 ** 2)

        entrada = torch.randn(
            1, 3, 640, 640,
            device=device,
            dtype=torch.float32
        )

        # Aquecimento
        with torch.inference_mode():
            for _ in range(30):
                _ = rede(entrada)

        torch.cuda.synchronize()

        tempos_ms = []

        # Cinco blocos de 100 inferências
        with torch.inference_mode():
            for bloco in range(5):

                torch.cuda.synchronize()
                inicio = time.perf_counter()

                for _ in range(100):
                    _ = rede(entrada)

                torch.cuda.synchronize()
                fim = time.perf_counter()

                latencia_media = ((fim - inicio) / 100) * 1000
                tempos_ms.append(latencia_media)

        latencia_mediana = float(np.median(tempos_ms))
        fps = 1000 / latencia_mediana

        resultados_yolo.append({
            "Modelo": nome_modelo,
            "Seed": seed,
            "Parametros": parametros,
            "Tamanho_MB": tamanho_mb,
            "Latencia_Mediana_ms": latencia_mediana,
            "FPS": fps
        })

        print(f"Parâmetros: {parametros:,}")
        print(f"Tamanho: {tamanho_mb:.3f} MB")
        print(f"Latência: {latencia_mediana:.3f} ms")
        print(f"FPS: {fps:.2f}")

        del rede, modelo, entrada
        torch.cuda.empty_cache()

df_yolo_eficiencia = pd.DataFrame(resultados_yolo)

display(df_yolo_eficiencia.round(4))

# Resumo por arquitetura
resumo_yolo_eficiencia = (
    df_yolo_eficiencia
    .groupby("Modelo")
    [["Parametros", "Tamanho_MB", "Latencia_Mediana_ms", "FPS"]]
    .agg(["mean", "std"])
    .round(4)
)

print("\nRESUMO POR ARQUITETURA")
display(resumo_yolo_eficiencia)

# Salvar resultados
PASTA = Path("/content/avaliacoes_finais/yolo")
PASTA.mkdir(parents=True, exist_ok=True)

df_yolo_eficiencia.to_csv(
    PASTA / "eficiencia_yolo_fundidos.csv",
    index=False,
    encoding="utf-8-sig"
)

resumo_yolo_eficiencia.to_csv(
    PASTA / "resumo_eficiencia_yolo_fundidos.csv",
    encoding="utf-8-sig"
)

print("\nTotal de checkpoints medidos:", len(df_yolo_eficiencia))
print("Total de inferências:", len(df_yolo_eficiencia) * 500)


## Gerar Matrizes de Confusão YOLO

In [ ]:
pip install ultralytics

In [ ]:
# @title

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from ultralytics import YOLO

RAIZ = Path("/content/todos_experimentos")
PASTA_SAIDA = Path("/content/avaliacoes_finais/yolo/matrizes_confusao")
PASTA_CONSOLIDADA = PASTA_SAIDA / "consolidadas_por_modelo"
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
PASTA_CONSOLIDADA.mkdir(parents=True, exist_ok=True)

matrizes_por_modelo = {}
nomes_classes_por_modelo = {}

for _, linha in df_yolo_melhor_f1.iterrows():
    nome_modelo = str(linha["Modelo"])
    seed = int(linha["Seed"])
    melhor_conf = float(linha["Melhor_Conf"])

    print(f"\nAvaliando: {nome_modelo} | Seed {seed} | conf={melhor_conf}")

    checkpoint = RAIZ / f"{nome_modelo}_seed_{seed}" / "weights" / "best.pt"
    modelo = YOLO(str(checkpoint))

    pasta_experimento = PASTA_SAIDA / f"{nome_modelo}_seed_{seed}"
    pasta_experimento.mkdir(parents=True, exist_ok=True)

    metricas = modelo.val(
        data="/content/dataset/data.yaml",
        split="test",
        imgsz=640,
        batch=16,
        conf=melhor_conf,
        device=0,
        plots=True,
        save_json=False,
        project=str(PASTA_SAIDA),
        name=f"{nome_modelo}_seed_{seed}",
        exist_ok=True,
        verbose=False
    )

    # Captura a matriz de confusão da seed atual e acumula por modelo
    cm = np.asarray(metricas.confusion_matrix.matrix, dtype=np.int64)
    labels = list(metricas.names.values()) + ["background"]

    if nome_modelo not in matrizes_por_modelo:
        matrizes_por_modelo[nome_modelo] = cm.copy()
        nomes_classes_por_modelo[nome_modelo] = labels
    else:
        matrizes_por_modelo[nome_modelo] += cm

print("\nGerando matrizes consolidadas (soma de todas as seeds por modelo)...")

for nome_modelo, cm_soma in matrizes_por_modelo.items():
    labels = nomes_classes_por_modelo[nome_modelo]

    # Salva a matriz somada em CSV
    df_cm = pd.DataFrame(cm_soma, index=labels, columns=labels)
    csv_path = PASTA_CONSOLIDADA / f"matriz_confusao_soma_{nome_modelo}.csv"
    df_cm.to_csv(csv_path)

    print(f"\n=== {nome_modelo} (Soma das seeds) ===")
    print(df_cm)

    # Plota e salva a matriz somada em PNG
    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm_soma,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels
    )
    plt.title(f"Matriz de Confusão Consolidada (Todas as Seeds) – {nome_modelo}")
    plt.xlabel("Predito")
    plt.ylabel("Real")
    plt.tight_layout()

    img_path = PASTA_CONSOLIDADA / f"matriz_confusao_soma_{nome_modelo}.png"
    plt.savefig(img_path, dpi=300)
    plt.close()

    print(f"Salvo em: {csv_path} e {img_path}")

print("\nGeração e consolidação das matrizes concluída.")

## ANOVA, Tukey e tamanho de efeito dos modelos YOLO

In [ ]:
# @title

import pandas as pd
import numpy as np
from scipy.stats import f_oneway, shapiro, levene
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Métricas principais para comparação
metricas = ["mAP50", "mAP50-95"]

modelos = ["yolov8n", "yolov9t", "yolov10n"]

resultados_anova = []
resultados_tukey = []

for metrica in metricas:

    print("\n" + "=" * 65)
    print(f"ANÁLISE DA MÉTRICA: {metrica}")
    print("=" * 65)

    grupos = [
        df_yolo_final.loc[
            df_yolo_final["Modelo"] == modelo, metrica
        ].dropna().values
        for modelo in modelos
    ]

    # Verificar quantidade de sementes
    for modelo, grupo in zip(modelos, grupos):
        print(f"{modelo}: {len(grupo)} observações")

    # Teste de normalidade por grupo
    print("\nTESTE DE SHAPIRO-WILK")
    for modelo, grupo in zip(modelos, grupos):
        estatistica, p = shapiro(grupo)
        print(f"{modelo}: p-valor = {p:.4f}")

    # Teste de homogeneidade das variâncias
    estat_levene, p_levene = levene(*grupos)
    print(f"\nLEVENe: p-valor = {p_levene:.4f}")

    # ANOVA de uma via
    estat_f, p_anova = f_oneway(*grupos)

    # Cálculo do eta-quadrado
    valores = np.concatenate(grupos)
    media_geral = np.mean(valores)

    ss_total = np.sum((valores - media_geral) ** 2)

    ss_entre = sum(
        len(grupo) * (np.mean(grupo) - media_geral) ** 2
        for grupo in grupos
    )

    eta2 = ss_entre / ss_total if ss_total > 0 else np.nan

    print("\nRESULTADO DA ANOVA")
    print(f"Estatística F: {estat_f:.4f}")
    print(f"p-valor: {p_anova:.6f}")
    print(f"Eta-quadrado: {eta2:.4f}")

    resultados_anova.append({
        "Metrica": metrica,
        "F": estat_f,
        "p_valor": p_anova,
        "Eta_quadrado": eta2,
        "p_Levene": p_levene
    })

    # Teste de Tukey
    dados_tukey = pd.DataFrame({
        "Modelo": np.repeat(modelos, [len(g) for g in grupos]),
        "Valor": valores
    })

    tukey = pairwise_tukeyhsd(
        endog=dados_tukey["Valor"],
        groups=dados_tukey["Modelo"],
        alpha=0.05
    )

    print("\nTESTE DE TUKEY")
    print(tukey)

    tabela_tukey = pd.DataFrame(
        data=tukey._results_table.data[1:],
        columns=tukey._results_table.data[0]
    )
    tabela_tukey.insert(0, "Metrica", metrica)
    resultados_tukey.append(tabela_tukey)

# Consolidar resultados
df_anova_yolo = pd.DataFrame(resultados_anova)
df_tukey_yolo = pd.concat(resultados_tukey, ignore_index=True)

# Salvar resultados
PASTA = "/content/avaliacoes_finais/yolo"

df_anova_yolo.to_csv(
    f"{PASTA}/anova_yolo.csv", index=False
)

df_tukey_yolo.to_csv(
    f"{PASTA}/tukey_yolo.csv", index=False
)

print("\nARQUIVOS SALVOS COM SUCESSO!")
print(df_anova_yolo)


## Estatísticas descritivas e comparação dos modelos YOLO.

In [ ]:
# @title

metricas = [
    "mAP50",
    "mAP50-95",
    "F1_With_Helmet",
    "F1_Without_Helmet"
]

df_resumo_yolo = (
    df_yolo_final
    .groupby("Modelo")[metricas]
    .agg(["mean", "std", "median"])
    .round(4)
)

display(df_resumo_yolo)

# Ranking descritivo baseado na média do mAP
df_ranking_yolo = (
    df_yolo_final
    .groupby("Modelo")
    .agg(
        mAP50_medio=("mAP50", "mean"),
        mAP50_std=("mAP50", "std"),
        mAP50_95_medio=("mAP50-95", "mean"),
        mAP50_95_std=("mAP50-95", "std")
    )
    .sort_values("mAP50_95_medio", ascending=False)
    .round(4)
)

print("\nRANKING DESCRITIVO — mAP50-95")
display(df_ranking_yolo)

PASTA = "/content/avaliacoes_finais/yolo"

df_ranking_yolo.to_csv(
    f"{PASTA}/ranking_descritivo_yolo.csv"
)

print("Ranking salvo com sucesso!")


# Falsos Positivos

In [ ]:
# @title
from pathlib import Path
import pandas as pd
import numpy as np
from ultralytics import YOLO
from PIL import Image

# Caminhos
PASTA_TESTE = Path("/content/dataset_tratado/test")
PASTA_IMAGENS = PASTA_TESTE / "images"
PASTA_ROTULOS = PASTA_TESTE / "labels"

PESO = "/content/todos_experimentos/todos_experimentos/yolov8n_seed_42/weights/best.pt"

PASTA_SAIDA = Path("/content/avaliacoes_finais")
PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

# Parâmetros
CONF = 0.001
IOU_MATCH = 0.45

# Carregar modelo
modelo = YOLO(PESO)

resultados = []

imagens = sorted(PASTA_IMAGENS.glob("*"))

print("Total de imagens:", len(imagens))

for caminho_img in imagens:

    if caminho_img.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
        continue

    resultado = modelo.predict(
        source=str(caminho_img),
        conf=CONF,
        iou=0.7,
        device=0,
        verbose=False
    )[0]

    largura = resultado.orig_shape[1]
    altura = resultado.orig_shape[0]

    caminho_txt = PASTA_ROTULOS / (caminho_img.stem + ".txt")

    gt_boxes = []

    if caminho_txt.exists():
        for linha in caminho_txt.read_text().splitlines():
            dados = linha.split()

            if len(dados) != 5:
                continue

            classe, xc, yc, w, h = map(float, dados)

            x1 = (xc - w / 2) * largura
            y1 = (yc - h / 2) * altura
            x2 = (xc + w / 2) * largura
            y2 = (yc + h / 2) * altura

            gt_boxes.append([x1, y1, x2, y2, int(classe)])

    if resultado.boxes is None:
        continue

    for box in resultado.boxes:

        classe = int(box.cls[0].item())

        # Investigar somente a classe Without Helmet
        if classe != 1:
            continue

        confianca = float(box.conf[0].item())
        x1, y1, x2, y2 = box.xyxy[0].cpu().numpy()

        maior_iou = 0
        classe_gt = None

        for gt in gt_boxes:

            gx1, gy1, gx2, gy2, c_gt = gt

            ix1 = max(x1, gx1)
            iy1 = max(y1, gy1)
            ix2 = min(x2, gx2)
            iy2 = min(y2, gy2)

            inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)

            area_pred = (x2 - x1) * (y2 - y1)
            area_gt = (gx2 - gx1) * (gy2 - gy1)

            uniao = area_pred + area_gt - inter

            iou = inter / uniao if uniao > 0 else 0

            if iou > maior_iou:
                maior_iou = iou
                classe_gt = c_gt

        if maior_iou < IOU_MATCH:

            resultados.append({
                "Imagem": caminho_img.name,
                "Caminho": str(caminho_img),
                "Confianca": confianca,
                "Maior_IoU_com_anotacao": maior_iou,
                "Classe_anotacao_mais_proxima": classe_gt,
                "X1": float(x1),
                "Y1": float(y1),
                "X2": float(x2),
                "Y2": float(y2)
            })

# Salvar CSV
df = pd.DataFrame(resultados)

arquivo_csv = PASTA_SAIDA / "investigacao_fp_yolov8n_seed42.csv"
df.to_csv(arquivo_csv, index=False)

print("\nInvestigação concluída!")
print("Total de detecções suspeitas:", len(df))
print("Total de imagens envolvidas:", df["Imagem"].nunique() if len(df) else 0)
print("CSV salvo em:", arquivo_csv)

display(df.head(10))


## Executar investigação automática

In [ ]:
# @title

from pathlib import Path
import pandas as pd
import cv2
from ultralytics import YOLO

# ==================================================
# CONFIGURAÇÕES
# ==================================================

PASTA_TESTE = Path("/content/dataset_tratado")
PASTA_IMAGENS = PASTA_TESTE / "images"
PASTA_ROTULOS = PASTA_TESTE / "labels"

PESO = "/content/todos_experimentos/todos_experimentos/yolov8n_seed_42/weights/best.pt"

PASTA_SAIDA = Path("/content/avaliacoes_finais/investigacao_automatica")
PASTA_VISUALIZACOES = PASTA_SAIDA / "imagens"

PASTA_VISUALIZACOES.mkdir(parents=True, exist_ok=True)

CONF = 0.001
IOU_NMS = 0.7
IOU_MATCH = 0.45

# ==================================================
# FUNÇÃO PARA CALCULAR IoU
# ==================================================

def calcular_iou(box1, box2):

    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersecao = max(0, x2-x1) * max(0, y2-y1)

    area1 = max(0, box1[2]-box1[0]) * max(0, box1[3]-box1[1])
    area2 = max(0, box2[2]-box2[0]) * max(0, box2[3]-box2[1])

    uniao = area1 + area2 - intersecao

    return intersecao / uniao if uniao > 0 else 0


# ==================================================
# CARREGAR MODELO
# ==================================================

modelo = YOLO(PESO)

imagens = sorted([
    p for p in PASTA_IMAGENS.iterdir()
    if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
])

registros = []

print("Imagens encontradas:", len(imagens))

# ==================================================
# INVESTIGAR TODAS AS IMAGENS
# ==================================================

for indice, caminho_img in enumerate(imagens, start=1):

    imagem = cv2.imread(str(caminho_img))

    if imagem is None:
        print("Não foi possível abrir:", caminho_img.name)
        continue

    altura, largura = imagem.shape[:2]

    caminho_txt = PASTA_ROTULOS / (caminho_img.stem + ".txt")

    anotacoes = []

    if caminho_txt.exists():

        for linha in caminho_txt.read_text().splitlines():

            dados = linha.split()

            if len(dados) != 5:
                continue

            classe, xc, yc, w, h = map(float, dados)

            box = [
                (xc - w/2) * largura,
                (yc - h/2) * altura,
                (xc + w/2) * largura,
                (yc + h/2) * altura
            ]

            anotacoes.append({
                "classe": int(classe),
                "box": box
            })

    # Inferência
    resultado = modelo.predict(
        source=str(caminho_img),
        conf=CONF,
        iou=IOU_NMS,
        device=0,
        verbose=False
    )[0]

    visualizacao = imagem.copy()

    # Desenhar anotações existentes em verde
    for anotacao in anotacoes:

        x1, y1, x2, y2 = map(int, anotacao["box"])

        cv2.rectangle(
            visualizacao,
            (x1, y1), (x2, y2),
            (0, 255, 0), 2
        )

        nome = "With Helmet" if anotacao["classe"] == 0 else "Without Helmet"

        cv2.putText(
            visualizacao, nome,
            (x1, max(20, y1-5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5, (0, 255, 0), 2
        )

    quantidade_suspeitas = 0

    if resultado.boxes is not None:

        for box in resultado.boxes:

            classe = int(box.cls[0].item())

            # Investigar somente Without Helmet
            if classe != 1:
                continue

            confianca = float(box.conf[0].item())
            coordenadas = box.xyxy[0].cpu().numpy().tolist()

            # Comparar com todas as anotações
            maior_iou_geral = 0
            maior_iou_classe1 = 0
            maior_iou_classe0 = 0

            for anotacao in anotacoes:

                iou = calcular_iou(
                    coordenadas,
                    anotacao["box"]
                )

                maior_iou_geral = max(maior_iou_geral, iou)

                if anotacao["classe"] == 1:
                    maior_iou_classe1 = max(maior_iou_classe1, iou)

                elif anotacao["classe"] == 0:
                    maior_iou_classe0 = max(maior_iou_classe0, iou)

            if maior_iou_classe1 >= IOU_MATCH:

                classificacao = "Correspondente a anotacao existente"
                cor = (255, 255, 0)

            elif maior_iou_classe0 >= IOU_MATCH:

                classificacao = "Possivel divergencia de classe"
                cor = (0, 165, 255)

            else:

                classificacao = "Possivel rotulo ausente ou falso positivo"
                cor = (0, 0, 255)

            quantidade_suspeitas += 1

            x1, y1, x2, y2 = map(int, coordenadas)

            cv2.rectangle(
                visualizacao,
                (x1, y1), (x2, y2),
                cor, 2
            )

            cv2.putText(
                visualizacao,
                f"Without Helmet: {confianca:.3f}",
                (x1, min(altura-5, y2+18)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5, cor, 2
            )

            registros.append({
                "Imagem": caminho_img.name,
                "Caminho": str(caminho_img),
                "Confianca": confianca,
                "IoU_com_qualquer_anotacao": maior_iou_geral,
                "IoU_com_classe_Without_Helmet": maior_iou_classe1,
                "IoU_com_classe_With_Helmet": maior_iou_classe0,
                "Classificacao_automatica": classificacao,
                "Revisao_manual": "Pendente",
                "Aprovado": ""
            })

    # Salvar visualização quando houver detecções da classe 1
    if quantidade_suspeitas > 0:

        cv2.imwrite(
            str(PASTA_VISUALIZACOES / caminho_img.name),
            visualizacao
        )

    print(
        f"[{indice}/{len(imagens)}] "
        f"{caminho_img.name} - "
        f"{quantidade_suspeitas} detecções"
    )

# ==================================================
# SALVAR RESULTADOS
# ==================================================

df = pd.DataFrame(registros)

arquivo_csv = PASTA_SAIDA / "resultado_investigacao.csv"

df.to_csv(
    arquivo_csv,
    index=False,
    encoding="utf-8-sig"
)

print("\n========== RESULTADO ==========")
print("Imagens processadas:", len(imagens))
print("Detecções de Without Helmet:", len(df))

if len(df) > 0:

    print("\nClassificações automáticas:")
    print(df["Classificacao_automatica"].value_counts())

print("\nCSV:", arquivo_csv)
print("Imagens:", PASTA_VISUALIZACOES)
print("\nNenhuma anotação original foi modificada.")


## Imagens com possiveis classes divergentes e faltantes

In [ ]:
# @title

import pandas as pd

arquivo = "/content/avaliacoes_finais/investigacao_automatica/resultado_investigacao.csv"

df = pd.read_csv(arquivo)

# Filtrar divergências e possíveis rótulos ausentes
filtro = df["Classificacao_automatica"].isin([
    "Possivel divergencia de classe",
    "Possivel rotulo ausente ou falso positivo"
])

divergencias = df[filtro].copy()

print("Total de ocorrências:", len(divergencias))
print("Total de imagens:", divergencias["Imagem"].nunique())

print("\nQuantidade por classificação:")
print(divergencias["Classificacao_automatica"].value_counts())

print("\nLista de imagens:")
display(
    divergencias[
        ["Imagem", "Confianca", "Classificacao_automatica"]
    ].sort_values("Imagem")
)


## Salvar em lista separada


In [ ]:
# @title

arquivo_saida = "/content/avaliacoes_finais/investigacao_automatica/imagens_para_revisar.csv"

divergencias.to_csv(
    arquivo_saida,
    index=False,
    encoding="utf-8-sig"
)

print("Lista salva em:", arquivo_saida)


## Visualizar imagens

In [ ]:
# @title

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from IPython.display import display

arquivo = "/content/avaliacoes_finais/investigacao_automatica/resultado_investigacao.csv"

df = pd.read_csv(arquivo)

# Filtrar somente os casos que precisam de revisão
filtro = df["Classificacao_automatica"].isin([
    "Possivel divergencia de classe",
    "Possivel rotulo ausente ou falso positivo"
])

revisar = df[filtro].copy()

print("Total de ocorrências:", len(revisar))
print("Total de imagens:", revisar["Imagem"].nunique())

# Visualizar uma imagem por vez
for nome_imagem, grupo in revisar.groupby("Imagem", sort=False):

    caminho = grupo.iloc[0]["Caminho"]
    caminho_visualizacao = (
        Path("/content/avaliacoes_finais/investigacao_automatica/imagens")
        / nome_imagem
    )

    if not caminho_visualizacao.exists():
        print("Imagem de visualização não encontrada:", nome_imagem)
        continue

    imagem = Image.open(caminho_visualizacao).convert("RGB")

    plt.figure(figsize=(14, 9))
    plt.imshow(imagem)
    plt.axis("off")
    plt.title(
        f"{nome_imagem}\n"
        f"Ocorrências suspeitas: {len(grupo)}",
        fontsize=11
    )
    plt.show()

    print("NOME DA IMAGEM:", nome_imagem)
    print("CAMINHO ORIGINAL:", caminho)

    print("\nDetecções:")
    display(
        grupo[
            [
                "Confianca",
                "IoU_com_qualquer_anotacao",
                "Classificacao_automatica"
            ]
        ].reset_index(drop=True)
    )

    print("-" * 80)


### Durante a revisão das anotações do conjunto de teste, foram identificadas e adicionadas **21 anotações ausentes**, referentes a objetos que estavam presentes nas imagens, mas não haviam sido inicialmente rotulados. Essa correção buscou tornar a avaliação dos modelos mais consistente com o conteúdo real das imagens.

# Teste de Inferência qualitativa (YOLO)

In [ ]:
pip install -q ultralytics

In [ ]:

from ultralytics import YOLO
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image
import torch

# ==========================================
# 1. Configurações
# ==========================================

IMAGENS = {
    "Imagem 1 - Com capacete": "/content/imagens_teste_final/foto_capacete2.jpg",
    "Imagem 2 - Sem capacete": "/content/imagens_teste_final/foto_scapacete2.jpg",
    "Imagem 3": "/content/imagens_teste_final/imagem_test2.jpg",
    "Imagem 4": "/content/imagens_teste_final/imagem_teste3.jpg",
    "Imagem 5": "/content/imagens_teste_final/imagem_teste4.jpg",
    "Imagem 6": "/content/imagens_teste_final/imagem_teste5.jpg"
}

MODELOS = ["yolov8n", "yolov9t", "yolov10n"]
SEED = 42

RAIZ = Path("/content/seed_42")
SAIDA = Path("/content/avaliacoes_finais/yolo/comparativo_individual")
SAIDA.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else "cpu"
CONF = 0.25
IMGSZ = 640

# ==========================================
# 2. Carregar modelos da seed 42
# ==========================================

modelos_carregados = {}

for nome in MODELOS:

    checkpoint = (
        RAIZ / f"{nome}_seed_{SEED}" / "weights" / "best.pt"
    )

    assert checkpoint.exists(), f"Checkpoint não encontrado: {checkpoint}"

    modelos_carregados[nome] = YOLO(str(checkpoint))

# ==========================================
# 3. Comparação individual por imagem
# ==========================================

for numero, (nome_imagem, caminho) in enumerate(IMAGENS.items(), start=1):

    caminho = Path(caminho)

    assert caminho.exists(), f"Imagem não encontrada: {caminho}"

    print(f"\n{'=' * 65}")
    print(nome_imagem.upper())
    print(f"{'=' * 65}")

    fig, eixos = plt.subplots(1, 3, figsize=(18, 6))

    for eixo, nome_modelo in zip(eixos, MODELOS):

        modelo = modelos_carregados[nome_modelo]

        resultado = modelo.predict(
            source=str(caminho),
            conf=CONF,
            imgsz=IMGSZ,
            device=DEVICE,
            verbose=False
        )[0]

        imagem_resultado = resultado.plot()

        imagem_resultado = Image.fromarray(
            imagem_resultado[..., ::-1]
        )

        eixo.imshow(imagem_resultado)
        eixo.set_title(nome_modelo, fontsize=14, fontweight="bold")
        eixo.axis("off")

        # Salvar resultado individual
        arquivo = f"imagem_{numero}_{nome_modelo}.jpg"

        imagem_resultado.save(
            SAIDA / arquivo,
            quality=95
        )

    fig.suptitle(
        nome_imagem,
        fontsize=17,
        fontweight="bold"
    )

    plt.tight_layout()
    plt.savefig(
        SAIDA / f"comparativo_imagem_{numero}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

print("\nComparações concluídas!")
print("Resultados salvos em:", SAIDA)


## Resultados dos experimentos (teste de inferência qualitativa)

In [ ]:
# ==========================================
# Comparação individual por imagem
# ==========================================

NOMES_CLASSES = {
    0: "With Helmet",
    1: "Without Helmet"
}

for numero, (nome_imagem, caminho) in enumerate(IMAGENS.items(), start=1):

    caminho = Path(caminho)

    assert caminho.exists(), f"Imagem não encontrada: {caminho}"

    print(f"\n{'=' * 80}")
    print(nome_imagem.upper())
    print(f"{'=' * 80}")

    fig, eixos = plt.subplots(1, 3, figsize=(18, 6))

    for eixo, nome_modelo in zip(eixos, MODELOS):

        modelo = modelos_carregados[nome_modelo]

        resultado = modelo.predict(
            source=str(caminho),
            conf=CONF,
            imgsz=IMGSZ,
            device=DEVICE,
            verbose=False
        )[0]

        print(f"\n--- {nome_modelo.upper()} ---")

        if len(resultado.boxes) == 0:
            print("Nenhuma detecção encontrada.")

        else:
            for i, caixa in enumerate(resultado.boxes, start=1):

                classe = int(caixa.cls[0].item())
                confianca = float(caixa.conf[0].item())

                x1, y1, x2, y2 = caixa.xyxy[0].tolist()

                nome_classe = NOMES_CLASSES.get(
                    classe,
                    f"Classe {classe}"
                )

                print(f"\nDetecção {i}")
                print(f"Classe predita: {nome_classe}")
                print(f"Confiança: {confianca:.4f} ({confianca * 100:.2f}%)")
                print(
                    f"Bounding Box: "
                    f"x1={x1:.1f}, y1={y1:.1f}, "
                    f"x2={x2:.1f}, y2={y2:.1f}"
                )

        imagem_resultado = resultado.plot()

        imagem_resultado = Image.fromarray(
            imagem_resultado[..., ::-1]
        )

        eixo.imshow(imagem_resultado)
        eixo.set_title(nome_modelo, fontsize=14, fontweight="bold")
        eixo.axis("off")

        arquivo = f"imagem_{numero}_{nome_modelo}.jpg"

        imagem_resultado.save(
            SAIDA / arquivo,
            quality=95
        )

    fig.suptitle(
        nome_imagem,
        fontsize=17,
        fontweight="bold"
    )

    plt.tight_layout()

    plt.savefig(
        SAIDA / f"comparativo_imagem_{numero}.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()
    plt.close(fig)

print("\nComparações concluídas!")
print("Resultados salvos em:", SAIDA)

# Treinar modelos complementares.

### Instalação e importações

In [ ]:
# @title

!pip install -q pycocotools

import os
import random
import time
import gc
import json
import numpy as np
import pandas as pd

from pathlib import Path
from PIL import Image
from functools import partial

import torch
import torchvision
import torchvision.transforms.functional as TF

from torch.utils.data import Dataset, DataLoader

from torchvision.models import (
    mobilenet_v2,
    MobileNet_V2_Weights
)

from torchvision.models.detection import (
    FasterRCNN,
    fasterrcnn_mobilenet_v3_large_fpn,
    ssdlite320_mobilenet_v3_large,
    FasterRCNN_MobileNet_V3_Large_FPN_Weights,
    SSDLite320_MobileNet_V3_Large_Weights
)

from torchvision.models.detection.faster_rcnn import FastRCNNPredictor

from torchvision.models.detection.ssdlite import (
    SSDLiteClassificationHead
)

from torchvision.models.detection.backbone_utils import BackboneWithFPN

from torchvision.models.detection.rpn import AnchorGenerator

from torchvision.ops import MultiScaleRoIAlign

from torchvision.ops.feature_pyramid_network import LastLevelMaxPool

print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())


### Conectar google drive

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')

### Configurações gerais


In [ ]:
# @title

# Dataset
DATASET = Path("/content/dataset_tratado")

TRAIN_IMG = DATASET / "train" / "images"
TRAIN_LBL = DATASET / "train" / "labels"

VALID_IMG = DATASET / "valid" / "images"
VALID_LBL = DATASET / "valid" / "labels"

TEST_IMG = DATASET / "test" / "images"
TEST_LBL = DATASET / "test" / "labels"

# Pasta de resultados
PASTA_EXPERIMENTOS = Path(
    "/content/drive/MyDrive/experimentos_torchvision_v2"
)

PASTA_EXPERIMENTOS.mkdir(
    parents=True,
    exist_ok=True
)

# Modelos
MODELOS = [
    "ssdlite_mobilenetv3"
]

# Seeds
SEEDS = [42, 123, 2025, 7, 21]

# Treinamento
NUM_CLASSES = 3  # Background + 2 classes

EPOCHS = 150
PATIENCE = 40

BATCH_EFETIVO = 16
BATCH_FISICO = 2
ACUMULACAO = BATCH_EFETIVO // BATCH_FISICO

IMG_SIZE = 640
WORKERS = 2

LEARNING_RATE = 0.005
MOMENTUM = 0.9
WEIGHT_DECAY = 0.0005

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

print("Dispositivo:", DEVICE)
print("Modelos:", MODELOS)
print("Seeds:", SEEDS)
print("Épocas máximas:", EPOCHS)
print("Patience:", PATIENCE)
print("Batch efetivo:", BATCH_EFETIVO)


### Verificação dos caminhos do dataset

In [ ]:
# @title

for caminho in [
    TRAIN_IMG, TRAIN_LBL,
    VALID_IMG, VALID_LBL,
    TEST_IMG, TEST_LBL
]:
    if not caminho.exists():
        raise FileNotFoundError(
            f"Caminho não encontrado: {caminho}"
        )

def contar_imagens(pasta):
    extensoes = [".jpg", ".jpeg", ".png", ".bmp"]
    return sum(
        1 for p in pasta.iterdir()
        if p.suffix.lower() in extensoes
    )

print("Dataset localizado!\n")

print("Treinamento:", contar_imagens(TRAIN_IMG))
print("Validação:", contar_imagens(VALID_IMG))
print("Teste:", contar_imagens(TEST_IMG))


### Controle de seeds

In [ ]:
# @title

def fixar_seed(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


def inicializar_worker(worker_id):

    seed = torch.initial_seed() % (2**32)

    np.random.seed(seed)
    random.seed(seed)


### Preparação do dataset

In [ ]:
# @title

class HelmetDataset(Dataset):

    def __init__(self, image_dir, label_dir, augment=False):

        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.augment = augment

        extensoes = [".jpg", ".jpeg", ".png", ".bmp"]

        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in extensoes
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):

        image_path = self.images[idx]

        label_path = self.label_dir / (
            image_path.stem + ".txt"
        )

        image = Image.open(image_path).convert("RGB")

        width, height = image.size

        boxes = []
        labels = []

        if label_path.exists():

            with open(label_path, "r") as f:
                linhas = f.readlines()

            for linha in linhas:

                valores = linha.strip().split()

                if len(valores) != 5:
                    continue

                classe, xc, yc, w, h = map(float, valores)

                if int(classe) not in [0, 1]:
                    continue

                xmin = (xc - w / 2) * width
                ymin = (yc - h / 2) * height

                xmax = (xc + w / 2) * width
                ymax = (yc + h / 2) * height

                xmin = max(0, min(xmin, width))
                ymin = max(0, min(ymin, height))

                xmax = max(0, min(xmax, width))
                ymax = max(0, min(ymax, height))

                if xmax <= xmin or ymax <= ymin:
                    continue

                boxes.append([
                    xmin, ymin, xmax, ymax
                ])

                labels.append(int(classe) + 1)

        boxes = torch.as_tensor(
            boxes,
            dtype=torch.float32
        ).reshape(-1, 4)

        labels = torch.as_tensor(
            labels,
            dtype=torch.int64
        )

        # Aumento de dados: espelhamento horizontal
        if self.augment and random.random() < 0.5:

            image = TF.hflip(image)

            if len(boxes) > 0:

                xmin = width - boxes[:, 2]
                xmax = width - boxes[:, 0]

                boxes[:, 0] = xmin
                boxes[:, 2] = xmax

        image = TF.to_tensor(image)

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": (
                (boxes[:, 2] - boxes[:, 0]) *
                (boxes[:, 3] - boxes[:, 1])
            ),
            "iscrowd": torch.zeros(
                (len(boxes),),
                dtype=torch.int64
            )
        }

        return image, target


def collate_fn(batch):
    return tuple(zip(*batch))


### Datasets e DataLoaders

In [ ]:
# @title

train_dataset = HelmetDataset(
    TRAIN_IMG,
    TRAIN_LBL,
    augment=True
)

valid_dataset = HelmetDataset(
    VALID_IMG,
    VALID_LBL,
    augment=False
)


def criar_loaders(seed):

    generator = torch.Generator()
    generator.manual_seed(seed)

    train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_FISICO,
    shuffle=True,
    num_workers=WORKERS,
    collate_fn=collate_fn,
    drop_last=True
)

    valid_loader = DataLoader(
        valid_dataset,
        batch_size=BATCH_FISICO,
        shuffle=False,
        num_workers=WORKERS,
        pin_memory=True,
        collate_fn=collate_fn
    )

    return train_loader, valid_loader


print("Dataset de treinamento:", len(train_dataset))
print("Dataset de validação:", len(valid_dataset))


### Criação das arquiteturas

In [ ]:
# @title

def criar_modelo(nome):

    # FASTER R-CNN + MOBILENETV2 + FPN
    if nome == "fasterrcnn_mobilenetv2":

        backbone_original = mobilenet_v2(
            weights=MobileNet_V2_Weights.DEFAULT
        ).features

        backbone = BackboneWithFPN(
            backbone_original,

            return_layers={
                "3": "0",
                "6": "1",
                "13": "2",
                "18": "3"
            },

            in_channels_list=[
                24, 32, 96, 1280
            ],

            out_channels=256,

            extra_blocks=LastLevelMaxPool()
        )

        anchor_generator = AnchorGenerator(
            sizes=(
                (32,),
                (64,),
                (128,),
                (256,),
                (512,)
            ),

            aspect_ratios=(
                (0.5, 1.0, 2.0),
            ) * 5
        )

        roi_pooler = MultiScaleRoIAlign(
            featmap_names=["0", "1", "2", "3"],
            output_size=7,
            sampling_ratio=2
        )

        model = FasterRCNN(
            backbone,
            num_classes=NUM_CLASSES,
            rpn_anchor_generator=anchor_generator,
            box_roi_pool=roi_pooler,
            min_size=IMG_SIZE,
            max_size=IMG_SIZE
        )

    # FASTER R-CNN + MOBILENETV3
    elif nome == "fasterrcnn_mobilenetv3":

        model = fasterrcnn_mobilenet_v3_large_fpn(
            weights=FasterRCNN_MobileNet_V3_Large_FPN_Weights.DEFAULT
        )

        in_features = (
            model.roi_heads.box_predictor.cls_score.in_features
        )

        model.roi_heads.box_predictor = FastRCNNPredictor(
            in_features,
            NUM_CLASSES
        )

        model.transform.min_size = (IMG_SIZE,)
        model.transform.max_size = IMG_SIZE

    # SSDlite + MOBILENETV3
    elif nome == "ssdlite_mobilenetv3":

        model = ssdlite320_mobilenet_v3_large(
            weights=SSDLite320_MobileNet_V3_Large_Weights.DEFAULT
        )

        in_channels = model.backbone.out_channels

        num_anchors = (
            model.anchor_generator.num_anchors_per_location()
        )

        model.head.classification_head = SSDLiteClassificationHead(
            in_channels,
            num_anchors,
            NUM_CLASSES,
            norm_layer=partial(
                torch.nn.BatchNorm2d,
                eps=0.001,
                momentum=0.03
            )
        )

    else:
        raise ValueError(f"Modelo desconhecido: {nome}")

    return model.to(DEVICE)


### Funções de treinamento e validação


In [ ]:
# @title

def treinar_epoca(model, loader, optimizer):

    model.train()

    total_loss = 0.0
    total_batches = len(loader)

    optimizer.zero_grad(set_to_none=True)

    for i, (images, targets) in enumerate(loader):

        images = [
            img.to(DEVICE, non_blocking=True)
            for img in images
        ]

        targets = [
            {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in t.items()
            }
            for t in targets
        ]

        loss_dict = model(images, targets)

        loss = sum(v for v in loss_dict.values())

        inicio_grupo = (
            (i // ACUMULACAO) * ACUMULACAO
        )

        tamanho_grupo = min(
            ACUMULACAO,
            total_batches - inicio_grupo
        )

        (loss / tamanho_grupo).backward()

        total_loss += loss.item()

        if (
            (i + 1) % ACUMULACAO == 0
            or (i + 1) == total_batches
        ):

            optimizer.step()
            optimizer.zero_grad(set_to_none=True)

    return total_loss / max(total_batches, 1)


def validar_epoca(model, loader):

    # Necessário para obter as losses no Torchvision.
    model.train()

    # Congelar estatísticas de BatchNorm na validação.
    for module in model.modules():

        if isinstance(
            module,
            torch.nn.modules.batchnorm._BatchNorm
        ):

            module.eval()

    total_loss = 0.0
    total_batches = 0

    with torch.no_grad():

        for images, targets in loader:

            images = [
                img.to(DEVICE, non_blocking=True)
                for img in images
            ]

            targets = [
                {
                    k: v.to(DEVICE, non_blocking=True)
                    for k, v in t.items()
                }
                for t in targets
            ]

            loss_dict = model(images, targets)

            loss = sum(v for v in loss_dict.values())

            total_loss += loss.item()
            total_batches += 1

    return total_loss / max(total_batches, 1)


### Função de treinamento de cada experimento

In [ ]:
# @title

def treinar_experimento(nome_modelo, seed):

    print("\n" + "=" * 65)
    print(f"MODELO: {nome_modelo}")
    print(f"SEED: {seed}")
    print("=" * 65)

    fixar_seed(seed)

    pasta = (
        PASTA_EXPERIMENTOS /
        f"{nome_modelo}_seed_{seed}"
    )

    pasta.mkdir(parents=True, exist_ok=True)

    pasta_pesos = pasta / "weights"
    pasta_pesos.mkdir(exist_ok=True)

    train_loader, valid_loader = criar_loaders(seed)

    model = criar_modelo(nome_modelo)

    optimizer = torch.optim.SGD(
        model.parameters(),
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = torch.optim.lr_scheduler.StepLR(
        optimizer,
        step_size=50,
        gamma=0.1
    )

    melhor_loss = float("inf")
    melhor_epoca = 0
    epocas_sem_melhora = 0

    historico = []

    inicio_total = time.time()

    for epoca in range(1, EPOCHS + 1):

        inicio_epoca = time.time()

        loss_treino = treinar_epoca(
            model,
            train_loader,
            optimizer
        )

        loss_validacao = validar_epoca(
            model,
            valid_loader
        )

        scheduler.step()

        lr_atual = optimizer.param_groups[0]["lr"]

        tempo_epoca = time.time() - inicio_epoca

        historico.append({
            "modelo": nome_modelo,
            "seed": seed,
            "epoca": epoca,
            "train_loss": loss_treino,
            "val_loss": loss_validacao,
            "learning_rate": lr_atual,
            "tempo_segundos": tempo_epoca
        })

        print(
            f"Época [{epoca:03d}/{EPOCHS}] | "
            f"Train Loss: {loss_treino:.4f} | "
            f"Val Loss: {loss_validacao:.4f} | "
            f"LR: {lr_atual:.6f} | "
            f"Tempo: {tempo_epoca:.1f}s"
        )

        # Salvar melhor modelo
        if loss_validacao < melhor_loss:

            melhor_loss = loss_validacao
            melhor_epoca = epoca
            epocas_sem_melhora = 0

            torch.save(
                {
                    "modelo": nome_modelo,
                    "seed": seed,
                    "epoca": epoca,
                    "model_state_dict": model.state_dict(),
                    "val_loss": melhor_loss
                },
                pasta_pesos / "best.pt"
            )

            print("  Melhor modelo salvo.")

        else:

            epocas_sem_melhora += 1

        # Salvar último checkpoint
        torch.save(
            {
                "modelo": nome_modelo,
                "seed": seed,
                "epoca": epoca,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "melhor_loss": melhor_loss,
                "melhor_epoca": melhor_epoca
            },
            pasta_pesos / "last.pt"
        )

        # Early stopping
        if epocas_sem_melhora >= PATIENCE:

            print(
                f"\nEarly stopping na época {epoca}."
            )

            break

    tempo_total = time.time() - inicio_total

    # Salvar histórico
    df_historico = pd.DataFrame(historico)

    df_historico.to_csv(
        pasta / "historico.csv",
        index=False
    )

    resultado = {
        "modelo": nome_modelo,
        "seed": seed,
        "melhor_epoca": melhor_epoca,
        "melhor_val_loss": melhor_loss,
        "epocas_executadas": len(historico),
        "tempo_total_minutos": tempo_total / 60,
        "status": "concluido"
    }

    with open(
        pasta / "resumo.json",
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            resultado,
            f,
            indent=4,
            ensure_ascii=False
        )

    print("\nExperimento finalizado.")
    print("Melhor época:", melhor_epoca)
    print("Melhor Val Loss:", round(melhor_loss, 5))
    print(
        "Tempo total:",
        round(tempo_total / 60, 2),
        "minutos"
    )

    del model
    del optimizer
    del scheduler
    del train_loader
    del valid_loader

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return resultado


### Executar os 15 treinamentos

In [ ]:
# @title

resultados = []

for nome_modelo in MODELOS:

    for seed in SEEDS:

        resultado = treinar_experimento(
            nome_modelo,
            seed
        )

        resultados.append(resultado)

        # Atualizar resumo geral após cada experimento
        pd.DataFrame(resultados).to_csv(
            PASTA_EXPERIMENTOS / "resumo_treinamentos.csv",
            index=False
        )

print("\nTODOS OS EXPERIMENTOS FINALIZADOS!")


# Avaliação dos modelos complementares.

## Verificar checkpoints torchvision

In [ ]:
# @title
# Conectar google drive

from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# @title

from pathlib import Path
import pandas as pd

PASTA_TORCH = Path(
    "/content/drive/MyDrive/experimentos_torchvision_v2"
)

MODELOS_TORCH = [
    "fasterrcnn_mobilenetv2",
    "fasterrcnn_mobilenetv3",
    "ssdlite_mobilenetv3"
]

SEEDS = [42, 123, 2025, 7, 21]

resultados = []

for modelo in MODELOS_TORCH:
    for seed in SEEDS:

        checkpoint = (
            PASTA_TORCH /
            f"{modelo}_seed_{seed}_v2" /
            "weights" /
            "best.pt"
        )

        resultados.append({
            "Modelo": modelo,
            "Seed": seed,
            "Checkpoint": str(checkpoint),
            "Encontrado": checkpoint.exists(),
            "Tamanho_MB": (
                round(checkpoint.stat().st_size / (1024 ** 2), 2)
                if checkpoint.exists() else None
            )
        })

df_checkpoints_torch = pd.DataFrame(resultados)

display(df_checkpoints_torch)

print("\nRESUMO")
print("Encontrados:", df_checkpoints_torch["Encontrado"].sum())
print("Ausentes:", (~df_checkpoints_torch["Encontrado"]).sum())


## Preparar avaliação

In [ ]:
# @title

from pathlib import Path
import torch
import numpy as np
import pandas as pd
import gc
import time

# Dataset corrigido
PASTA_DATASET = Path("/content/dataset_tratado")
PASTA_TESTE = PASTA_DATASET / "test"

# Experimentos
PASTA_TORCH = Path(
    "/content/drive/MyDrive/experimentos_torchvision_v2"
)

# Resultados
PASTA_AVALIACOES_TORCH = Path(
    "/content/avaliacoes_finais/torchvision"
)
PASTA_AVALIACOES_TORCH.mkdir(parents=True, exist_ok=True)

# Modelos e sementes
MODELOS_TORCH = [
    "fasterrcnn_mobilenetv2",
    "fasterrcnn_mobilenetv3",
    "ssdlite_mobilenetv3"
]

SEEDS = [42, 123, 2025, 7, 21]

# Classes
NOME_CLASSES = {
    1: "With Helmet",
    2: "Without Helmet"
}

# Parâmetros
IMG_SIZE = 640
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Limiares
CONF_MAP = 0.001
CONF_PR = 0.25
IOU_THRESHOLD = 0.50

# Arquivos de teste
imagens_teste = sorted(
    list((PASTA_TESTE / "images").glob("*.jpg")) +
    list((PASTA_TESTE / "images").glob("*.jpeg")) +
    list((PASTA_TESTE / "images").glob("*.png"))
)

print("Configuração concluída.")
print("Dispositivo:", DEVICE)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Imagens de teste:", len(imagens_teste))
print("Modelos:", MODELOS_TORCH)
print("Sementes:", SEEDS)
print("Total de avaliações:", len(MODELOS_TORCH) * len(SEEDS))
print("Confiança para mAP:", CONF_MAP)
print("Confiança inicial para P/R:", CONF_PR)


## Preparar as funções de carregamento

In [ ]:
# @title
import torch
import torch.nn as nn

from torchvision.models import (
    mobilenet_v2,
    mobilenet_v3_large,
    MobileNet_V2_Weights,
    MobileNet_V3_Large_Weights
)

from torchvision.models.detection import (
    fasterrcnn_mobilenet_v3_large_fpn,
    ssdlite320_mobilenet_v3_large
)

from torchvision.models.detection.backbone_utils import (
    BackboneWithFPN
)

NUM_CLASSES = 3  # Background + 2 classes


def criar_modelo_torch(nome):

    # Faster R-CNN + MobileNetV2
    if nome == "fasterrcnn_mobilenetv2":

        backbone_original = mobilenet_v2(
            weights=None
        ).features

        backbone = BackboneWithFPN(
            backbone_original,

            return_layers={
                "3": "0",
                "6": "1",
                "13": "2",
                "18": "3"
            },

            in_channels_list=[
                24,
                32,
                96,
                1280
            ],

            out_channels=256
        )

        from torchvision.models.detection import FasterRCNN

        modelo = FasterRCNN(
            backbone,
            num_classes=NUM_CLASSES
        )

    # Faster R-CNN + MobileNetV3
    elif nome == "fasterrcnn_mobilenetv3":

        modelo = fasterrcnn_mobilenet_v3_large_fpn(
            weights=None,
            weights_backbone=None,
            num_classes=NUM_CLASSES
        )

    # SSDlite + MobileNetV3
    elif nome == "ssdlite_mobilenetv3":

        modelo = ssdlite320_mobilenet_v3_large(
            weights=None,
            weights_backbone=None,
            num_classes=NUM_CLASSES
        )

    else:
        raise ValueError(f"Modelo desconhecido: {nome}")

    return modelo


print("Arquiteturas Torchvision atualizadas.")


def carregar_checkpoint(nome, seed):

    caminho = (
        PASTA_TORCH /
        f"{nome}_seed_{seed}_v2" /
        "weights" /
        "best.pt"
    )

    modelo = criar_modelo_torch(nome)

    checkpoint = torch.load(
        caminho,
        map_location="cpu",
        weights_only=False
    )

    # Aceitar checkpoint puro ou dicionário contendo state_dict
    if isinstance(checkpoint, dict):
        if "model_state_dict" in checkpoint:
            estado = checkpoint["model_state_dict"]
        elif "state_dict" in checkpoint:
            estado = checkpoint["state_dict"]
        else:
            estado = checkpoint
    else:
        estado = checkpoint

    modelo.load_state_dict(estado)
    modelo.to(DEVICE)
    modelo.eval()

    return modelo


print("Funções de carregamento definidas.")

## Construção das arquiteturas Torchvision para avaliação

In [ ]:
# @title

from functools import partial

from torchvision.models import mobilenet_v2
from torchvision.models.detection import (
    FasterRCNN,
    fasterrcnn_mobilenet_v3_large_fpn,
    ssdlite320_mobilenet_v3_large
)
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.ops import MultiScaleRoIAlign
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.ssdlite import SSDLiteClassificationHead


def criar_modelo_torch(nome):

    # --------------------------------------------------
    # 1. Faster R-CNN MobileNetV2
    # --------------------------------------------------

    if nome == "fasterrcnn_mobilenetv2":

        backbone_original = mobilenet_v2(
            weights=None
        ).features

        backbone = BackboneWithFPN(
            backbone_original,
            return_layers={
                "3": "0",
                "6": "1",
                "13": "2",
                "18": "3"
            },
            in_channels_list=[24, 32, 96, 1280],
            out_channels=256,
            extra_blocks=LastLevelMaxPool()
        )

        anchor_generator = AnchorGenerator(
            sizes=(
                (32,),
                (64,),
                (128,),
                (256,),
                (512,)
            ),
            aspect_ratios=((0.5, 1.0, 2.0),) * 5
        )

        roi_pooler = MultiScaleRoIAlign(
            featmap_names=["0", "1", "2", "3"],
            output_size=7,
            sampling_ratio=2
        )

        modelo = FasterRCNN(
            backbone,
            num_classes=NUM_CLASSES,
            rpn_anchor_generator=anchor_generator,
            box_roi_pool=roi_pooler,
            min_size=IMG_SIZE,
            max_size=IMG_SIZE
        )

    # --------------------------------------------------
    # 2. Faster R-CNN MobileNetV3
    # --------------------------------------------------

    elif nome == "fasterrcnn_mobilenetv3":

        modelo = fasterrcnn_mobilenet_v3_large_fpn(
            weights=None,
            weights_backbone=None
        )

        in_features = (
            modelo.roi_heads.box_predictor.cls_score.in_features
        )

        modelo.roi_heads.box_predictor = FastRCNNPredictor(
            in_features,
            NUM_CLASSES
        )

        modelo.transform.min_size = (IMG_SIZE,)
        modelo.transform.max_size = IMG_SIZE

    # --------------------------------------------------
    # 3. SSDlite MobileNetV3
    # --------------------------------------------------

    elif nome == "ssdlite_mobilenetv3":

       modelo = ssdlite320_mobilenet_v3_large(
        weights=None,
        weights_backbone=None,
        num_classes=NUM_CLASSES
    )

    else:
        raise ValueError(f"Modelo desconhecido: {nome}")

    return modelo.to(DEVICE)


## Preparação dos DataLoaders Torchvision

In [ ]:
# @title
import random
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TF
from PIL import Image

# Definição do Dataset para garantir que seja encontrado na execução atual
class HelmetDataset(Dataset):
    def __init__(self, image_dir, label_dir, augment=False):
        self.image_dir = Path(image_dir)
        self.label_dir = Path(label_dir)
        self.augment = augment
        extensoes = [".jpg", ".jpeg", ".png", ".bmp"]
        self.images = sorted([
            p for p in self.image_dir.iterdir()
            if p.suffix.lower() in extensoes
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        image_path = self.images[idx]
        label_path = self.label_dir / (image_path.stem + ".txt")
        image = Image.open(image_path).convert("RGB")
        width, height = image.size
        boxes = []
        labels = []

        if label_path.exists():
            with open(label_path, "r") as f:
                linhas = f.readlines()
            for linha in linhas:
                valores = linha.strip().split()
                if len(valores) != 5:
                    continue
                classe, xc, yc, w, h = map(float, valores)
                if int(classe) not in [0, 1]:
                    continue
                xmin = (xc - w / 2) * width
                ymin = (yc - h / 2) * height
                xmax = (xc + w / 2) * width
                ymax = (yc + h / 2) * height
                xmin = max(0, min(xmin, width))
                ymin = max(0, min(ymin, height))
                xmax = max(0, min(xmax, width))
                ymax = max(0, min(ymax, height))
                if xmax <= xmin or ymax <= ymin:
                    continue
                boxes.append([xmin, ymin, xmax, ymax])
                labels.append(int(classe) + 1)

        boxes = torch.as_tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.as_tensor(labels, dtype=torch.int64)

        if self.augment and random.random() < 0.5:
            image = TF.hflip(image)
            if len(boxes) > 0:
                xmin = width - boxes[:, 2]
                xmax = width - boxes[:, 0]
                boxes[:, 0] = xmin
                boxes[:, 2] = xmax

        image = TF.to_tensor(image)
        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": torch.tensor([idx]),
            "area": ((boxes[:, 2] - boxes[:, 0]) * (boxes[:, 3] - boxes[:, 1])),
            "iscrowd": torch.zeros((len(boxes),), dtype=torch.int64)
        }
        return image, target

def collate_fn(batch):
    return tuple(zip(*batch))

# Diretórios do dataset corrigido
PASTA_DATASET = Path("/content/dataset_tratado")

# Conjuntos de validação e teste
dataset_valid = HelmetDataset(
    image_dir=PASTA_DATASET / "valid/images",
    label_dir=PASTA_DATASET / "valid/labels",
    augment=False
)

dataset_test = HelmetDataset(
    image_dir=PASTA_DATASET / "test/images",
    label_dir=PASTA_DATASET / "test/labels",
    augment=False
)

# DataLoaders
loader_valid = DataLoader(
    dataset_valid,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

loader_test = DataLoader(
    dataset_test,
    batch_size=4,
    shuffle=False,
    num_workers=2,
    collate_fn=collate_fn
)

print("DATASETS PREPARADOS")
print("=" * 45)
print("Imagens de validação:", len(dataset_valid))
print("Imagens de teste:", len(dataset_test))
print("Lotes de validação:", len(loader_valid))
print("Lotes de teste:", len(loader_test))

## Preparação do Ground Truth COCO

In [ ]:
# @title

from pycocotools.coco import COCO

CATEGORIAS_COCO = [
    {"id": 1, "name": "With Helmet"},
    {"id": 2, "name": "Without Helmet"}
]

def criar_coco_ground_truth(dataset):

    imagens = []
    anotacoes = []
    contador = 1

    for idx in range(len(dataset)):

        image, target = dataset[idx]
        height, width = image.shape[1:]

        imagens.append({
            "id": idx,
            "width": width,
            "height": height,
            "file_name": dataset.images[idx].name
        })

        for box, label, area in zip(
            target["boxes"],
            target["labels"],
            target["area"]
        ):

            xmin, ymin, xmax, ymax = box.tolist()

            anotacoes.append({
                "id": contador,
                "image_id": idx,
                "category_id": int(label),
                "bbox": [
                    xmin, ymin,
                    xmax - xmin,
                    ymax - ymin
                ],
                "area": float(area),
                "iscrowd": 0
            })

            contador += 1

    coco = COCO()
    coco.dataset = {
        "images": imagens,
        "annotations": anotacoes,
        "categories": CATEGORIAS_COCO
    }

    coco.createIndex()

    return coco


coco_valid = criar_coco_ground_truth(dataset_valid)
coco_test = criar_coco_ground_truth(dataset_test)

print("GROUND TRUTH PREPARADO")
print("Validação:", len(coco_valid.imgs), "imagens |",
      len(coco_valid.anns), "anotações")
print("Teste:", len(coco_test.imgs), "imagens |",
      len(coco_test.anns), "anotações")


## Funções de avaliação Torchvision

In [ ]:
# @title

import numpy as np
import torch
from pycocotools.cocoeval import COCOeval

# --------------------------------------------------
# 1. GERAR PREDIÇÕES
# --------------------------------------------------

@torch.no_grad()
def gerar_predicoes_coco(modelo, loader):
    modelo.eval()
    predicoes = []

    for imagens, alvos in loader:
        imagens = [img.to(DEVICE) for img in imagens]
        saidas = modelo(imagens)

        for alvo, saida in zip(alvos, saidas):
            image_id = int(alvo["image_id"].item())

            boxes = saida["boxes"].cpu().numpy()
            labels = saida["labels"].cpu().numpy()
            scores = saida["scores"].cpu().numpy()

            for box, label, score in zip(boxes, labels, scores):
                if score < CONF_MAP:
                    continue

                xmin, ymin, xmax, ymax = box.tolist()

                predicoes.append({
                    "image_id": image_id,
                    "category_id": int(label),
                    "bbox": [
                        xmin,
                        ymin,
                        xmax - xmin,
                        ymax - ymin
                    ],
                    "score": float(score)
                })

    return predicoes


# --------------------------------------------------
# 2. CALCULAR MAP PELO COCO
# --------------------------------------------------

def calcular_metricas_coco(coco_gt, predicoes):
    if not predicoes:
        return {
            "mAP50": 0.0,
            "mAP50-95": 0.0
        }

    coco_dt = coco_gt.loadRes(predicoes)

    avaliador = COCOeval(coco_gt, coco_dt, "bbox")
    avaliador.params.imgIds = coco_gt.getImgIds()
    avaliador.params.catIds = [1, 2]
    avaliador.params.maxDets = [1, 10, MAX_DETECTIONS]

    avaliador.evaluate()
    avaliador.accumulate()

    precisao = avaliador.eval["precision"]

    # Formato: IoU x Recall x Classe x Área x MaxDets
    precisao = precisao[:, :, :, 0, -1]

    precisao_valida = precisao[precisao >= 0]

    map_50_95 = (
        float(precisao_valida.mean())
        if len(precisao_valida) else 0.0
    )

    precisao_50 = precisao[0]
    precisao_50_valida = precisao_50[precisao_50 >= 0]

    map_50 = (
        float(precisao_50_valida.mean())
        if len(precisao_50_valida) else 0.0
    )

    return {
        "mAP50": map_50,
        "mAP50-95": map_50_95
    }


# --------------------------------------------------
# 3. PREPARAR CURVAS DE PRECISÃO E RECALL
# --------------------------------------------------

def preparar_curvas_pr(dataset, predicoes, iou_thr=0.50):
    dados = {}

    for classe in [1, 2]:
        dados[classe] = {
            "scores": [],
            "tp": [],
            "total_gt": 0
        }

    predicoes_por_imagem = {}

    for pred in predicoes:
        predicoes_por_imagem.setdefault(
            pred["image_id"], []
        ).append(pred)

    for idx in range(len(dataset)):
        _, alvo = dataset[idx]

        gt_boxes = alvo["boxes"].numpy()
        gt_labels = alvo["labels"].numpy()

        for classe in [1, 2]:
            indices_gt = np.where(gt_labels == classe)[0]
            boxes_gt = gt_boxes[indices_gt]

            dados[classe]["total_gt"] += len(boxes_gt)

            preds = [
                p for p in predicoes_por_imagem.get(idx, [])
                if p["category_id"] == classe
            ]

            preds.sort(
                key=lambda p: p["score"],
                reverse=True
            )

            boxes_pred = np.array([
                [
                    p["bbox"][0],
                    p["bbox"][1],
                    p["bbox"][0] + p["bbox"][2],
                    p["bbox"][1] + p["bbox"][3]
                ]
                for p in preds
            ], dtype=float).reshape(-1, 4)

            scores = [p["score"] for p in preds]
            usados = set()

            for box_pred, score in zip(boxes_pred, scores):
                melhor_iou = 0.0
                melhor_gt = -1

                for j, box_gt in enumerate(boxes_gt):
                    if j in usados:
                        continue

                    x1 = max(box_pred[0], box_gt[0])
                    y1 = max(box_pred[1], box_gt[1])
                    x2 = min(box_pred[2], box_gt[2])
                    y2 = min(box_pred[3], box_gt[3])

                    inter = max(0, x2-x1) * max(0, y2-y1)

                    area_pred = max(0, box_pred[2]-box_pred[0]) * max(0, box_pred[3]-box_pred[1])
                    area_gt = max(0, box_gt[2]-box_gt[0]) * max(0, box_gt[3]-box_gt[1])

                    uniao = area_pred + area_gt - inter
                    iou = inter / uniao if uniao > 0 else 0

                    if iou > melhor_iou:
                        melhor_iou = iou
                        melhor_gt = j

                verdadeiro_positivo = (
                    melhor_gt >= 0 and melhor_iou >= iou_thr
                )

                if verdadeiro_positivo:
                    usados.add(melhor_gt)

                dados[classe]["scores"].append(score)
                dados[classe]["tp"].append(
                    int(verdadeiro_positivo)
                )

    for classe in [1, 2]:
        scores = np.array(dados[classe]["scores"])
        tp = np.array(dados[classe]["tp"])

        ordem = np.argsort(-scores)

        dados[classe]["scores"] = scores[ordem]
        dados[classe]["tp_acumulado"] = np.cumsum(tp[ordem])

    return dados


# --------------------------------------------------
# 4. CALCULAR MÉTRICAS EM UM LIMIAR
# --------------------------------------------------

def metricas_no_limiar(dados, limiar):
    resultados = {}

    for classe in [1, 2]:
        scores = dados[classe]["scores"]
        tp_acumulado = dados[classe]["tp_acumulado"]
        total_gt = dados[classe]["total_gt"]

        quantidade = np.searchsorted(
            -scores, -limiar, side="right"
        )

        tp = (
            int(tp_acumulado[quantidade-1])
            if quantidade > 0 else 0
        )

        fp = quantidade - tp
        fn = total_gt - tp

        precisao = tp / (tp + fp) if tp + fp > 0 else 0
        recall = tp / (tp + fn) if tp + fn > 0 else 0

        f1 = (
            2 * precisao * recall / (precisao + recall)
            if precisao + recall > 0 else 0
        )

        resultados[classe] = {
            "Precisao": precisao,
            "Recall": recall,
            "F1": f1
        }

    resultados["F1_Medio"] = np.mean([
        resultados[1]["F1"],
        resultados[2]["F1"]
    ])

    return resultados


# --------------------------------------------------
# 5. SELECIONAR MELHOR LIMIAR NA VALIDAÇÃO
# --------------------------------------------------

def selecionar_melhor_limiar(dados):
    scores = np.concatenate([
        dados[1]["scores"],
        dados[2]["scores"]
    ])

    if len(scores) == 0:
        return 0.5, 0.0

    limiares = np.unique(scores)

    melhor_limiar = 0.5
    melhor_f1 = -1

    for limiar in limiares:
        resultado = metricas_no_limiar(dados, limiar)

        if resultado["F1_Medio"] > melhor_f1:
            melhor_f1 = resultado["F1_Medio"]
            melhor_limiar = float(limiar)

    return melhor_limiar, melhor_f1


print("Funções de avaliação preparadas.")


## Avaliação dos 15 experimentos Torchvision

In [ ]:
# @title
# CONFIGURAÇÕES #
MAX_DETECTIONS = 300
CONF_MAP = 0.001

print("Configurações atualizadas.")

In [ ]:
# @title
import pandas as pd
import gc
from pathlib import Path

PASTA_RESULTADOS = Path(
    "/content/avaliacoes_finais/torchvision"
)
PASTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 2025, 7, 21]

resultados_torch = []
arquivo_resultados = (
    PASTA_RESULTADOS / "resultados_torchvision_consolidados.csv"
)

for nome_modelo in MODELOS_TORCH:
    for seed in SEEDS:

        print("\n" + "=" * 65)
        print(f"MODELO: {nome_modelo} | SEED: {seed}")
        print("=" * 65)

        # Carregar modelo
        modelo = carregar_checkpoint(nome_modelo, seed)

        # Predições na validação
        pred_valid = gerar_predicoes_coco(
            modelo, loader_valid
        )

        curvas_valid = preparar_curvas_pr(
            dataset_valid, pred_valid
        )

        melhor_conf, f1_valid = selecionar_melhor_limiar(
            curvas_valid
        )

        # Predições no teste
        pred_test = gerar_predicoes_coco(
            modelo, loader_test
        )

        metricas_map = calcular_metricas_coco(
            coco_test, pred_test
        )

        curvas_test = preparar_curvas_pr(
            dataset_test, pred_test
        )

        metricas_test = metricas_no_limiar(
            curvas_test, melhor_conf
        )

        # Consolidar resultados
        linha = {
            "Modelo": nome_modelo,
            "Seed": seed,
            "mAP50": metricas_map["mAP50"],
            "mAP50-95": metricas_map["mAP50-95"],
            "Melhor_Conf_Validacao": melhor_conf,
            "F1_Medio_Validacao": f1_valid,

            "Precisao_With_Helmet":
                metricas_test[1]["Precisao"],
            "Recall_With_Helmet":
                metricas_test[1]["Recall"],
            "F1_With_Helmet":
                metricas_test[1]["F1"],

            "Precisao_Without_Helmet":
                metricas_test[2]["Precisao"],
            "Recall_Without_Helmet":
                metricas_test[2]["Recall"],
            "F1_Without_Helmet":
                metricas_test[2]["F1"],

            "F1_Medio_Teste":
                metricas_test["F1_Medio"]
        }

        resultados_torch.append(linha)

        # Salvar após cada experimento
        pd.DataFrame(resultados_torch).to_csv(
            arquivo_resultados,
            index=False,
            encoding="utf-8-sig"
        )

        print(f"mAP50: {linha['mAP50']:.4f}")
        print(f"mAP50-95: {linha['mAP50-95']:.4f}")
        print(f"Melhor confiança: {melhor_conf:.4f}")
        print(f"F1 médio no teste: {linha['F1_Medio_Teste']:.4f}")

        del modelo, pred_valid, pred_test
        del curvas_valid, curvas_test
        gc.collect()
        torch.cuda.empty_cache()

df_torch_final = pd.DataFrame(resultados_torch)

print("\nAVALIAÇÃO CONCLUÍDA")
display(df_torch_final.round(4))

print(f"\nArquivo salvo em: {arquivo_resultados}")

## Consolidação dos resultados

In [ ]:
# @title

import pandas as pd
from pathlib import Path

# Carregar resultados das duas famílias
df_yolo_final = pd.read_csv(
    "/content/avaliacoes_finais/yolo/resultados_yolo_consolidados.csv"
)

df_torch_final = pd.read_csv(
    "/content/avaliacoes_finais/torchvision/resultados_torchvision_consolidados.csv"
)

# Identificar as famílias
df_yolo_final["Familia"] = "YOLO"
df_torch_final["Familia"] = "Torchvision"

# Unificar os resultados
df_comparativo = pd.concat(
    [df_yolo_final, df_torch_final],
    ignore_index=True
)

# Salvar tabela completa
arquivo_comparativo = (
    "/content/avaliacoes_finais/resultados_comparativos.csv"
)

df_comparativo.to_csv(
    arquivo_comparativo,
    index=False,
    encoding="utf-8-sig"
)

# Resumo por arquitetura
metricas = [
    "mAP50",
    "mAP50-95",
    "F1_With_Helmet",
    "F1_Without_Helmet",
    "F1_Medio_Teste"
]

resumo = (
    df_comparativo
    .groupby(["Familia", "Modelo"])[metricas]
    .agg(["mean", "std"])
    .round(4)
)

print("TOTAL DE EXPERIMENTOS:", len(df_comparativo))
print("\nRESULTADOS MÉDIOS POR ARQUITETURA:")
display(resumo)

print("\nTabela completa salva em:")
print(arquivo_comparativo)


ANOVA, Tukey e testes complementares

In [ ]:
# @title

import pandas as pd
from scipy.stats import shapiro, levene, kruskal
from statsmodels.formula.api import ols
from statsmodels.stats.anova import anova_lm
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from statsmodels.stats.oneway import anova_oneway

df_stats = df_comparativo.copy()
df_stats = df_stats.rename(columns={"mAP50-95": "mAP50_95"})

metricas = ["mAP50", "mAP50_95", "F1_Medio_Teste"]

for metrica in metricas:
    print("\n" + "=" * 70)
    print(f"ANÁLISE ESTATÍSTICA: {metrica}")
    print("=" * 70)

    # ANOVA tradicional
    modelo = ols(f"{metrica} ~ C(Modelo)", data=df_stats).fit()
    tabela_anova = anova_lm(modelo, typ=2)

    print("\nANOVA:")
    display(tabela_anova)

    # Pressupostos
    grupos = [
        grupo[metrica].dropna().values
        for _, grupo in df_stats.groupby("Modelo")
    ]

    stat_shapiro, p_shapiro = shapiro(modelo.resid)
    stat_levene, p_levene = levene(*grupos)

    print(f"\nShapiro-Wilk: p = {p_shapiro:.6f}")
    print(f"Levene:       p = {p_levene:.6f}")

    # Tamanho do efeito
    eta2 = (
        tabela_anova.loc["C(Modelo)", "sum_sq"]
        / tabela_anova["sum_sq"].sum()
    )
    print(f"Eta²:         {eta2:.4f}")

    # ANOVA de Welch
    resultado_welch = anova_oneway(
        grupos,
        use_var="unequal"
    )
    print(
        f"\nWelch: F = {resultado_welch.statistic:.4f} | "
        f"p = {resultado_welch.pvalue:.6f}"
    )

    # Kruskal-Wallis
    resultado_kruskal = kruskal(*grupos)
    print(
        f"Kruskal-Wallis: H = {resultado_kruskal.statistic:.4f} | "
        f"p = {resultado_kruskal.pvalue:.6f}"
    )

    # Comparações múltiplas de Tukey
    tukey = pairwise_tukeyhsd(
        endog=df_stats[metrica],
        groups=df_stats["Modelo"],
        alpha=0.05
    )

    print("\nTukey HSD:")
    print(tukey)


## Matrizes de confusão — Torchvision

In [ ]:
# @title
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import defaultdict

PASTA_MATRIZES = Path("/content/avaliacoes_finais/torchvision/matrizes_confusao")
PASTA_MATRIZES.mkdir(parents=True, exist_ok=True)

NOMES_CLASSES = ["With Helmet", "Without Helmet", "Background"]

def calcular_iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    intersecao = max(0, x2 - x1) * max(0, y2 - y1)

    area1 = max(0, box1[2] - box1[0]) * max(0, box1[3] - box1[1])
    area2 = max(0, box2[2] - box2[0]) * max(0, box2[3] - box2[1])

    return intersecao / (area1 + area2 - intersecao + 1e-9)


def construir_matriz_confusao(dataset, predicoes, confianca, iou_min=0.5):
    matriz = np.zeros((3, 3), dtype=int)

    pred_por_imagem = defaultdict(list)
    for p in predicoes:
        if p["score"] >= confianca:
            pred_por_imagem[p["image_id"]].append(p)

    for i in range(len(dataset)):
        _, target = dataset[i]
        image_id = int(target["image_id"].item())

        boxes_gt = target["boxes"].cpu().numpy()
        labels_gt = target["labels"].cpu().numpy()

        preds = sorted(
            pred_por_imagem[image_id],
            key=lambda x: x["score"],
            reverse=True
        )

        boxes_pred = []
        for p in preds:
            x, y, w, h = p["bbox"]
            boxes_pred.append([
                x, y, x + w, y + h,
                int(p["category_id"])
            ])

        associados_gt = set()
        associados_pred = set()

        # Associação gulosa por IoU
        candidatos = []
        for j, pred in enumerate(boxes_pred):
            for k, gt in enumerate(boxes_gt):
                iou = calcular_iou(pred[:4], gt)
                if iou >= iou_min:
                    candidatos.append((iou, j, k))

        candidatos.sort(reverse=True)

        for iou, j, k in candidatos:
            if j in associados_pred or k in associados_gt:
                continue

            classe_pred = boxes_pred[j][4] - 1
            classe_real = int(labels_gt[k]) - 1

            if classe_real in (0, 1) and classe_pred in (0, 1):
                matriz[classe_real, classe_pred] += 1

            associados_gt.add(k)
            associados_pred.add(j)

        # Objetos reais não detectados: Background na predição
        for k, classe in enumerate(labels_gt):
            if k not in associados_gt:
                matriz[int(classe) - 1, 2] += 1

        # Detecções sem objeto correspondente: falso positivo
        for j, pred in enumerate(boxes_pred):
            if j not in associados_pred:
                classe_pred = pred[4] - 1
                if classe_pred in (0, 1):
                    matriz[2, classe_pred] += 1

    return matriz


matrizes_por_modelo = {}

for modelo_nome in MODELOS_TORCH:
    matriz_total = np.zeros((3, 3), dtype=int)

    for seed in [42, 123, 2025, 7, 21]:
        linha = df_torch_final[
            (df_torch_final["Modelo"] == modelo_nome) &
            (df_torch_final["Seed"] == seed)
        ]

        confianca = float(linha["Melhor_Conf_Validacao"].iloc[0])

        modelo = carregar_checkpoint(modelo_nome, seed)
        # Removida a chamada à 'configurar_avaliacao' que causava o erro,
        # pois carregar_checkpoint já coloca o modelo em modo de avaliação (.eval())

        predicoes = gerar_predicoes_coco(modelo, loader_test)

        matriz = construir_matriz_confusao(
            dataset_test,
            predicoes,
            confianca
        )

        matriz_total += matriz
        del modelo

    matrizes_por_modelo[modelo_nome] = matriz_total

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        matriz_total,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=NOMES_CLASSES,
        yticklabels=NOMES_CLASSES
    )
    plt.title(f"Matriz de confusão — {modelo_nome}")
    plt.xlabel("Classe predita")
    plt.ylabel("Classe real")
    plt.tight_layout()

    plt.savefig(
        PASTA_MATRIZES / f"{modelo_nome}_matriz_confusao.png",
        dpi=300
    )
    plt.show()
    plt.close()

print("Matrizes salvas em:", PASTA_MATRIZES)

## Teste estatístico Torchvision (ANOVA, Tukey)

In [ ]:
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 1. Carregar o arquivo CSV com os resultados finais
df = pd.read_csv('resultados_finais_completos.csv')

def analisar_familia(dataframe, familia_nome, metricas=['mAP50', 'mAP50-95']):
    df_fam = dataframe[dataframe['Familia'] == familia_nome].copy()
    resultados_anova = []

    print(f"\n{'#'*65}")
    print(f"FAMÍLIA: {familia_nome.upper()}")
    print(f"{'#'*65}")

    for metrica in metricas:
        print("\n" + "="*65)
        print(f"ANÁLISE DA MÉTRICA: {metrica}")
        print("="*65)

        grupos = []
        modelos_nomes = []

        for modelo, grupo in df_fam.groupby('Modelo'):
            valores = grupo[metrica].values
            grupos.append(valores)
            modelos_nomes.append(modelo)
            print(f"{modelo}: {len(valores)} observações")

        # Teste de normalidade (Shapiro-Wilk)
        print("\nTESTE DE SHAPIRO-WILK")
        for modelo, valores in zip(modelos_nomes, grupos):
            _, p_shapiro = stats.shapiro(valores)
            print(f"{modelo}: p-valor = {p_shapiro:.4f}")

        # Teste de homogeneidade de variâncias (Levene)
        _, p_levene = stats.levene(*grupos)
        print(f"\nLEVENE: p-valor = {p_levene:.4f}")

        # ANOVA de um fator
        f_stat, p_val = stats.f_oneway(*grupos)

        # Cálculo do Eta-quadrado (SS_entre / SS_total)
        media_geral = df_fam[metrica].mean()
        ss_entre = sum(len(g) * (g.mean() - media_geral)**2 for g in grupos)
        ss_total = sum((df_fam[metrica] - media_geral)**2)
        eta_quadrado = ss_entre / ss_total

        print("\nRESULTADO DA ANOVA")
        print(f"Estatística F: {f_stat:.4f}")
        print(f"p-valor: {p_val:.6f}")
        print(f"Eta-quadrado: {eta_quadrado:.4f}")

        # Teste post-hoc de Tukey
        print("\nTESTE DE TUKEY")
        tukey = pairwise_tukeyhsd(endog=df_fam[metrica], groups=df_fam['Modelo'], alpha=0.05)
        print(tukey)

        # Salvar tabela do Tukey em CSV
        tukey_df = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
        tukey_df.to_csv(f'tukey_{familia_nome.lower()}_{metrica}.csv', index=False)

        resultados_anova.append({
            'Familia': familia_nome,
            'Metrica': metrica,
            'F': f_stat,
            'p_valor': p_val,
            'Eta_quadrado': eta_quadrado,
            'p_Levene': p_levene
        })

    return pd.DataFrame(resultados_anova)

# 2. Executar para YOLO e Torchvision
res_yolo = analisar_familia(df, 'YOLO', metricas=['mAP50', 'mAP50-95'])
res_torchvision = analisar_familia(df, 'Torchvision', metricas=['mAP50', 'mAP50-95'])

# 3. Consolidar e salvar o resumo geral da ANOVA
resumo_geral = pd.concat([res_yolo, res_torchvision], ignore_index=True)
resumo_geral.to_csv('resumo_anova_familias.csv', index=False)

print("\nARQUIVOS SALVOS COM SUCESSO!")
print(resumo_geral)

## Complexidade computacional — Torchvision

In [ ]:
# @title
!pip install -q fvcore

In [ ]:
# @title
import os
import torch
import pandas as pd
from pathlib import Path
from fvcore.nn import FlopCountAnalysis

RAIZ_CHECKPOINTS = Path("/content/drive/MyDrive/experimentos_torchvision_v2")

resultados_complexidade = []

for nome in MODELOS_TORCH:
    # Carregar uma semente para medir a arquitetura
    modelo = carregar_checkpoint(nome, 42)
    modelo.eval()

    parametros = sum(p.numel() for p in modelo.parameters())
    parametros_treinaveis = sum(
        p.numel() for p in modelo.parameters() if p.requires_grad
    )

    # Estimativa de operações para uma imagem de entrada 640x640
    imagem = torch.rand(3, 640, 640, device=DEVICE)

    try:
        flops = FlopCountAnalysis(modelo, ([imagem],))
        macs = flops.total()
        gflops = (macs * 2) / 1e9
    except Exception as erro:
        print(f"Falha no cálculo de GFLOPs para {nome}: {erro}")
        gflops = float("nan")

    # Tamanho médio dos cinco checkpoints
    tamanhos = []

    for seed in [42, 123, 2025, 7, 21]:
        caminho = (
            RAIZ_CHECKPOINTS /
            f"{nome}_seed_{seed}_v2" /
            "weights" /
            "best.pt"
        )
        tamanhos.append(os.path.getsize(caminho) / (1024 ** 2))

    resultados_complexidade.append({
        "Modelo": nome,
        "Parametros_Totais": parametros,
        "Parametros_Treinaveis": parametros_treinaveis,
        "GFLOPs_estimados": gflops,
        "Tamanho_Medio_MB": sum(tamanhos) / len(tamanhos)
    })

    del modelo
    torch.cuda.empty_cache()

df_complexidade_torch = pd.DataFrame(resultados_complexidade)

display(df_complexidade_torch.round(4))

arquivo = "/content/avaliacoes_finais/torchvision/complexidade_modelos.csv"
df_complexidade_torch.to_csv(arquivo, index=False, encoding="utf-8-sig")

print("Arquivo salvo em:", arquivo)

## Latência e FPS

In [ ]:
# @title
import time
import torch
import pandas as pd
import numpy as np

resultados_tempo = []

torch.backends.cudnn.benchmark = True

for nome in MODELOS_TORCH:
    for seed in [42, 123, 2025, 7, 21]:
        print(f"Medindo {nome} | Seed {seed}")

        modelo = carregar_checkpoint(nome, seed)
        modelo.eval()

        imagem = torch.rand(3, 640, 640, device=DEVICE)

        # Aquecimento
        with torch.inference_mode():
            for _ in range(30):
                modelo([imagem])

        torch.cuda.synchronize()

        tempos_blocos = []

        # Cinco blocos de 100 execuções
        with torch.inference_mode():
            for bloco in range(5):
                tempos = []

                for _ in range(100):
                    torch.cuda.synchronize()
                    inicio = time.perf_counter()

                    modelo([imagem])

                    torch.cuda.synchronize()
                    fim = time.perf_counter()

                    tempos.append((fim - inicio) * 1000)

                tempos_blocos.append(np.mean(tempos))

        latencia = float(np.median(tempos_blocos))
        fps = 1000 / latencia

        resultados_tempo.append({
            "Modelo": nome,
            "Seed": seed,
            "Latencia_ms": latencia,
            "FPS": fps
        })

        del modelo
        torch.cuda.empty_cache()

df_tempo_torch = pd.DataFrame(resultados_tempo)

resumo_tempo_torch = (
    df_tempo_torch
    .groupby("Modelo")[["Latencia_ms", "FPS"]]
    .agg(["mean", "std"])
    .round(4)
)

display(resumo_tempo_torch)

arquivo = "/content/avaliacoes_finais/torchvision/latencia_fps.csv"
df_tempo_torch.to_csv(arquivo, index=False, encoding="utf-8-sig")

print("Arquivo salvo em:", arquivo)

# Consolidação final — comparação das seis arquiteturas

In [ ]:
# @title

import pandas as pd

# ==========================================
# 1. Complexidade dos modelos YOLO
# ==========================================

df_complexidade_yolo_final = (
    df_complexidade_yolo
    .groupby("Modelo")
    .agg(
        Parametros_Totais=("Parametros", "mean"),
        Tamanho_Medio_MB=("Tamanho_MB", "mean"),
        GFLOPs_estimados=("GFLOPs", "mean")
    )
    .reset_index()
)

# ==========================================
# 2. Complexidade dos modelos Torchvision
# ==========================================

df_complexidade_torch_final = df_complexidade_torch[
    ["Modelo", "Parametros_Totais",
     "GFLOPs_estimados", "Tamanho_Medio_MB"]
].copy()

df_complexidade_final = pd.concat(
    [df_complexidade_yolo_final, df_complexidade_torch_final],
    ignore_index=True
)

# ==========================================
# 3. Latência YOLO (somente rede, sem NMS)
# ==========================================

df_tempo_yolo_final = (
    df_yolo_eficiencia
    .groupby("Modelo")
    .agg(
        Latencia_YOLO_ms=("Latencia_Mediana_ms", "mean"),
        FPS_YOLO=("FPS", "mean")
    )
    .reset_index()
)

# ==========================================
# 4. Latência Torchvision (pipeline completo)
# ==========================================

df_tempo_torch_final = (
    df_tempo_torch
    .groupby("Modelo")
    .agg(
        Latencia_Torchvision_ms=("Latencia_ms", "mean"),
        FPS_Torchvision=("FPS", "mean")
    )
    .reset_index()
)

df_tempo_final = df_tempo_yolo_final.merge(
    df_tempo_torch_final,
    on="Modelo",
    how="outer"
)

# ==========================================
# 5. Unificar todos os resultados
# ==========================================

df_resultado_final = (
    df_comparativo
    .merge(df_complexidade_final, on="Modelo", how="left")
    .merge(df_tempo_final, on="Modelo", how="left")
)

# ==========================================
# 6. Resumo por arquitetura
# ==========================================

colunas = [
    "mAP50",
    "mAP50-95",
    "F1_With_Helmet",
    "F1_Without_Helmet",
    "F1_Medio_Teste",
    "Parametros_Totais",
    "GFLOPs_estimados",
    "Tamanho_Medio_MB",
    "Latencia_YOLO_ms",
    "FPS_YOLO",
    "Latencia_Torchvision_ms",
    "FPS_Torchvision"
]

resumo_final = (
    df_resultado_final
    .groupby(["Familia", "Modelo"])[colunas]
    .agg(["mean", "std"])
    .round(4)
)

display(resumo_final)

# ==========================================
# 7. Salvar arquivos
# ==========================================

PASTA = "/content/avaliacoes_finais"

arquivo_completo = f"{PASTA}/resultados_finais_completos.csv"
arquivo_resumo = f"{PASTA}/resumo_comparativo_final.csv"

df_resultado_final.to_csv(
    arquivo_completo,
    index=False,
    encoding="utf-8-sig"
)

resumo_final.to_csv(
    arquivo_resumo,
    encoding="utf-8-sig"
)

print("Resultados completos:", arquivo_completo)
print("Resumo comparativo:", arquivo_resumo)


# Teste de Inferência qualitativa (Torchvision)

In [ ]:

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
pip install -q torch torchvision

In [ ]:
import torch
import torchvision
from torchvision.models import mobilenet_v2
from torchvision.models.detection import (
    FasterRCNN,
    fasterrcnn_mobilenet_v3_large_fpn,
    ssdlite320_mobilenet_v3_large
)
from torchvision.models.detection.backbone_utils import BackboneWithFPN
from torchvision.models.detection.rpn import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.ops import MultiScaleRoIAlign
from torchvision.ops.feature_pyramid_network import LastLevelMaxPool

from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pandas as pd
import numpy as np

# Configurações
IMAGENS = {
    "Com capacete": "/content/imagens_teste_final/foto_capacete2.jpg",
    "Sem capacete": "/content/imagens_teste_final/foto_scapacete2.jpg",
    "Imagem 2": "/content/imagens_teste_final/imagem_test2.jpg",
    "Imagem 3": "/content/imagens_teste_final/imagem_teste3.jpg",
    "Imagem 4": "/content/imagens_teste_final/imagem_teste4.jpg",
    "Imagem 5": "/content/imagens_teste_final/imagem_teste5.jpg",
    }

MODELOS = [
    "fasterrcnn_mobilenetv2",
    "fasterrcnn_mobilenetv3",
    "ssdlite_mobilenetv3"
]

SEED = 42
NUM_CLASSES = 3
CONF = 0.25

DEVICE = torch.device(
    "cuda:0" if torch.cuda.is_available() else "cpu"
)

BASE_DIR = Path(
    "/content/drive/MyDrive/experimentos_torchvision_v2"
)

PASTA_SAIDA = Path(
    "/content/avaliacoes_finais/teste_externo_torchvision"
)

PASTA_SAIDA.mkdir(parents=True, exist_ok=True)

CLASSES = {
    1: "Com capacete",
    2: "Sem capacete"
}

# Construção das arquiteturas
def criar_modelo(nome):

    if nome == "fasterrcnn_mobilenetv2":

        backbone_base = mobilenet_v2(
            weights=None
        ).features

        backbone = BackboneWithFPN(
            backbone_base,
            return_layers={
                "3": "0",
                "6": "1",
                "13": "2",
                "18": "3"
            },
            in_channels_list=[24, 32, 96, 1280],
            out_channels=256,
            extra_blocks=LastLevelMaxPool()
        )

        anchor_generator = AnchorGenerator(
            sizes=((32,), (64,), (128,), (256,), (512,)),
            aspect_ratios=((0.5, 1.0, 2.0),) * 5
        )

        roi_pooler = MultiScaleRoIAlign(
            featmap_names=["0", "1", "2", "3"],
            output_size=7,
            sampling_ratio=2
        )

        modelo = FasterRCNN(
            backbone,
            num_classes=NUM_CLASSES,
            rpn_anchor_generator=anchor_generator,
            box_roi_pool=roi_pooler,
            min_size=640,
            max_size=640
        )

    elif nome == "fasterrcnn_mobilenetv3":

        modelo = fasterrcnn_mobilenet_v3_large_fpn(
            weights=None,
            weights_backbone=None
        )

        in_features = (
            modelo.roi_heads.box_predictor.cls_score.in_features
        )

        modelo.roi_heads.box_predictor = FastRCNNPredictor(
            in_features,
            NUM_CLASSES
        )

        modelo.transform.min_size = (640,)
        modelo.transform.max_size = 640

    elif nome == "ssdlite_mobilenetv3":

        modelo = ssdlite320_mobilenet_v3_large(
            weights=None,
            weights_backbone=None,
            num_classes=NUM_CLASSES
        )

    else:
        raise ValueError(f"Modelo desconhecido: {nome}")

    return modelo


# Carregar checkpoints
modelos_carregados = {}

for nome in MODELOS:

    caminho_peso = (
        BASE_DIR /
        f"{nome}_seed_{SEED}_v2" /
        "weights" /
        "best.pt"
    )

    if not caminho_peso.exists():
        raise FileNotFoundError(caminho_peso)

    modelo = criar_modelo(nome)

    checkpoint = torch.load(
        caminho_peso,
        map_location=DEVICE,
        weights_only=False
    )

    if "model_state_dict" in checkpoint:
        state_dict = checkpoint["model_state_dict"]
    elif "state_dict" in checkpoint:
        state_dict = checkpoint["state_dict"]
    elif "model_state" in checkpoint:
        state_dict = checkpoint["model_state"]
    else:
        state_dict = checkpoint

    modelo.load_state_dict(state_dict, strict=True)

    modelo.to(DEVICE)
    modelo.eval()

    modelos_carregados[nome] = modelo

    print(f"Modelo carregado: {nome}")


# Executar inferências
resultados = []

for situacao, caminho_imagem in IMAGENS.items():

    if not Path(caminho_imagem).exists():
        raise FileNotFoundError(caminho_imagem)

    imagem = Image.open(caminho_imagem).convert("RGB")

    imagem_tensor = (
        torchvision.transforms.functional.to_tensor(imagem)
        .to(DEVICE)
    )

    print(f"\n{'='*50}")
    print(f"IMAGEM: {situacao}")
    print(f"{'='*50}")

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))

    for i, nome in enumerate(MODELOS):

        modelo = modelos_carregados[nome]

        with torch.no_grad():
            predicao = modelo([imagem_tensor])[0]

        caixas = predicao["boxes"].cpu().numpy()
        scores = predicao["scores"].cpu().numpy()
        labels = predicao["labels"].cpu().numpy()

        # Filtrar detecções pelo limiar de confiança
        indices = scores >= CONF

        caixas = caixas[indices]
        scores = scores[indices]
        labels = labels[indices]

        # Exibir imagem e detecções
        axes[i].imshow(imagem)

        for caixa, score, label in zip(caixas, scores, labels):

            x1, y1, x2, y2 = caixa

            classe = CLASSES.get(int(label), f"Classe {label}")

            retangulo = patches.Rectangle(
                (x1, y1),
                x2 - x1,
                y2 - y1,
                linewidth=2,
                edgecolor="lime",
                facecolor="none"
            )

            axes[i].add_patch(retangulo)

            axes[i].text(
                x1,
                max(0, y1 - 5),
                f"{classe}: {score:.2f}",
                color="white",
                fontsize=9,
                bbox=dict(facecolor="green", alpha=0.8)
            )

            resultados.append({
                "Imagem": situacao,
                "Modelo": nome,
                "Classe prevista": classe,
                "Confiança": float(score)
            })

            print(
                f"{nome} | {classe} | "
                f"Confiança: {score:.4f}"
            )

        if len(caixas) == 0:
            print(f"{nome}: Nenhum objeto detectado.")

        axes[i].set_title(nome)
        axes[i].axis("off")

        # Salvar imagem individual
        pasta_modelo = (
            PASTA_SAIDA /
            nome /
            situacao.replace(" ", "_")
        )

        pasta_modelo.mkdir(parents=True, exist_ok=True)

        fig_individual, ax_individual = plt.subplots(figsize=(10, 7))
        ax_individual.imshow(imagem)

        for caixa, score, label in zip(caixas, scores, labels):
            x1, y1, x2, y2 = caixa
            classe = CLASSES.get(int(label), f"Classe {label}")

            ax_individual.add_patch(
                patches.Rectangle(
                    (x1, y1),
                    x2 - x1,
                    y2 - y1,
                    linewidth=2,
                    edgecolor="lime",
                    facecolor="none"
                )
            )

            ax_individual.text(
                x1, max(0, y1 - 5),
                f"{classe}: {score:.2f}",
                color="white",
                bbox=dict(facecolor="green", alpha=0.8)
            )

        ax_individual.axis("off")
        fig_individual.savefig(
            pasta_modelo / "resultado.jpg",
            bbox_inches="tight",
            dpi=150
        )
        plt.close(fig_individual)

    fig.suptitle(f"Comparação Torchvision - {situacao}")
    plt.tight_layout()
    plt.show()
    plt.close(fig)


# Consolidar resultados
df_externo = pd.DataFrame(resultados)

df_externo.to_csv(
    PASTA_SAIDA / "resultados_teste_externo.csv",
    index=False
)

display(df_externo)

print("\nAvaliação externa Torchvision concluída!")
print(f"Resultados salvos em: {PASTA_SAIDA}")

# Tratar imagens sem rótulo

In [ ]:
# @title
from pathlib import Path

dataset = Path("/content/dataset_tratado")

conjuntos = ["train", "valid", "test"]

for conjunto in conjuntos:

    pasta_imagens = dataset / conjunto / "images"
    pasta_labels = dataset / conjunto / "labels"

    imagens = list(pasta_imagens.glob("*"))
    labels = list(pasta_labels.glob("*.txt"))

    labels_vazias = 0
    labels_com_objetos = 0
    imagens_sem_label = 0

    for imagem in imagens:

        arquivo_label = pasta_labels / f"{imagem.stem}.txt"

        if not arquivo_label.exists():
            imagens_sem_label += 1

        elif arquivo_label.stat().st_size == 0:
            labels_vazias += 1

        else:
            labels_com_objetos += 1

    print(f"\n===== {conjunto.upper()} =====")
    print(f"Total de imagens: {len(imagens)}")
    print(f"Labels com objetos: {labels_com_objetos}")
    print(f"Labels vazias: {labels_vazias}")
    print(f"Imagens sem arquivo de label: {imagens_sem_label}")
    print(f"Total sem anotações: {labels_vazias + imagens_sem_label}")

In [ ]:
# @title
import random
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

dataset = Path("/content/dataset/content/dataset")

random.seed(42)

for conjunto in ["train", "valid", "test"]:

    pasta_imagens = dataset / conjunto / "images"
    pasta_labels = dataset / conjunto / "labels"

    imagens_sem_rotulo = []

    for imagem in pasta_imagens.iterdir():

        if imagem.suffix.lower() not in {
            ".jpg", ".jpeg", ".png", ".bmp", ".webp"
        }:
            continue

        label = pasta_labels / f"{imagem.stem}.txt"

        if not label.exists() or not label.read_text().strip():
            imagens_sem_rotulo.append(imagem)

    amostras = random.sample(
        imagens_sem_rotulo,
        min(12, len(imagens_sem_rotulo))
    )

    print(f"\n{conjunto.upper()} - {len(imagens_sem_rotulo)} imagens sem rótulos")

    if not amostras:
        continue

    fig, eixos = plt.subplots(3, 4, figsize=(16, 10))
    fig.suptitle(
        f"Imagens sem anotações - {conjunto.upper()}",
        fontsize=16
    )

    for ax in eixos.flat:
        ax.axis("off")

    for ax, caminho in zip(eixos.flat, amostras):

        imagem = Image.open(caminho).convert("RGB")

        ax.imshow(imagem)
        ax.set_title(caminho.name, fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# @title
# Remover imagens sem rótulo, classe 'license' foi removida anteriormente.

from pathlib import Path

dataset = Path("/content/dataset/content/dataset")

extensoes = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

total_removidas = 0

for conjunto in ["train", "valid", "test"]:

    pasta_imagens = dataset / conjunto / "images"
    pasta_labels = dataset / conjunto / "labels"

    removidas = 0

    for imagem in pasta_imagens.iterdir():

        if imagem.suffix.lower() not in extensoes:
            continue

        label = pasta_labels / f"{imagem.stem}.txt"

        if not label.exists() or not label.read_text().strip():

            imagem.unlink()

            if label.exists():
                label.unlink()

            removidas += 1

    total_removidas += removidas

    print(f"{conjunto}: {removidas} imagens removidas")

print(f"\nTotal de imagens removidas: {total_removidas}")

## Verificar duplicatas

In [ ]:
# @title
!pip install -q ImageHash

In [ ]:
# @title
import imagehash
from PIL import Image
from itertools import combinations

registros = []

for conjunto, imagens in imagens_por_conjunto.items():

    for caminho in imagens:

        try:
            imagem = Image.open(caminho).convert("RGB")
            hash_phash = imagehash.phash(imagem)

            registros.append({
                "conjunto": conjunto,
                "arquivo": caminho.name,
                "caminho": str(caminho),
                "hash": int(str(hash_phash), 16)
            })

        except Exception as erro:
            print("Erro:", caminho, erro)

duplicatas_phash = []

# Comparar somente imagens de conjuntos diferentes
for a, b in combinations(registros, 2):

    if a["conjunto"] == b["conjunto"]:
        continue

    distancia = (a["hash"] ^ b["hash"]).bit_count()

    if distancia <= 8:

        duplicatas_phash.append({
            "Conjunto A": a["conjunto"],
            "Imagem A": a["arquivo"],
            "Caminho A": a["caminho"],
            "Conjunto B": b["conjunto"],
            "Imagem B": b["arquivo"],
            "Caminho B": b["caminho"],
            "Distância pHash": distancia
        })

import pandas as pd

tabela_phash = pd.DataFrame(duplicatas_phash)

print("Pares semelhantes encontrados:", len(tabela_phash))

display(tabela_phash)

In [ ]:
# @title
# Visualizar as possíveis duplicatas

import matplotlib.pyplot as plt
from PIL import Image

limite = min(20, len(tabela_phash))

for i in range(limite):

    linha = tabela_phash.iloc[i]

    img_a = Image.open(linha["Caminho A"]).convert("RGB")
    img_b = Image.open(linha["Caminho B"]).convert("RGB")

    fig, ax = plt.subplots(1, 2, figsize=(12, 5))

    ax[0].imshow(img_a)
    ax[0].set_title(
        f'{linha["Conjunto A"]}: {linha["Imagem A"]}'
    )

    ax[1].imshow(img_b)
    ax[1].set_title(
        f'{linha["Conjunto B"]}: {linha["Imagem B"]}'
    )

    for eixo in ax:
        eixo.axis("off")

    plt.suptitle(
        f'Distância pHash: {linha["Distância pHash"]}'
    )

    plt.tight_layout()
    plt.show()

In [ ]:
# @title
from collections import defaultdict

# Criar grupos de imagens conectadas pelas duplicatas
pai = {}

def encontrar(x):
    pai.setdefault(x, x)
    if pai[x] != x:
        pai[x] = encontrar(pai[x])
    return pai[x]

def unir(a, b):
    raiz_a = encontrar(a)
    raiz_b = encontrar(b)
    if raiz_a != raiz_b:
        pai[raiz_b] = raiz_a

for _, linha in tabela_phash.iterrows():
    unir(linha["Caminho A"], linha["Caminho B"])

grupos = defaultdict(list)

for caminho in pai:
    grupos[encontrar(caminho)].append(caminho)

grupos = [arquivos for arquivos in grupos.values() if len(arquivos) > 1]

# Prioridade: preservar test, depois valid, depois train
prioridade = {"test": 0, "valid": 1, "train": 2}

plano = []

for numero, grupo in enumerate(grupos, 1):
    grupo_ordenado = sorted(
        grupo,
        key=lambda x: prioridade[x.split("/images/")[0].split("/")[-1]]
    )

    preservar = grupo_ordenado[0]

    for caminho in grupo:
        conjunto = caminho.split("/images/")[0].split("/")[-1]

        plano.append({
            "Grupo": numero,
            "Conjunto": conjunto,
            "Imagem": caminho.split("/")[-1],
            "Ação": "PRESERVAR" if caminho == preservar else "REMOVER",
            "Caminho": caminho
        })

import pandas as pd

tabela_plano = pd.DataFrame(plano)
display(tabela_plano)

In [ ]:
# @title
from pathlib import Path

for _, linha in tabela_plano.iterrows():
    caminho_imagem = Path(linha["Caminho"])
    caminho_label = (
        caminho_imagem.parent.parent
        / "labels"
        / f"{caminho_imagem.stem}.txt"
    )

    print("\nGrupo:", linha["Grupo"])
    print("Imagem:", caminho_imagem.name)
    print("Ação:", linha["Ação"])

    if caminho_label.exists():
        conteudo = caminho_label.read_text().strip()
        print("Anotação:", conteudo if conteudo else "VAZIA")
    else:
        print("Anotação: ARQUIVO NÃO ENCONTRADO")

In [ ]:
# @title
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
from pathlib import Path

grupos_analisar = [1, 3, 4, 5, 6, 7]

for grupo in grupos_analisar:

    dados = tabela_plano[tabela_plano["Grupo"] == grupo]

    fig, axes = plt.subplots(1, len(dados), figsize=(14, 6))

    if len(dados) == 1:
        axes = [axes]

    for ax, (_, linha) in zip(axes, dados.iterrows()):

        caminho = Path(linha["Caminho"])
        imagem = Image.open(caminho).convert("RGB")

        largura, altura = imagem.size
        ax.imshow(imagem)

        label = (
            caminho.parent.parent
            / "labels"
            / f"{caminho.stem}.txt"
        )

        if label.exists():
            for linha_label in label.read_text().splitlines():

                valores = linha_label.split()

                if len(valores) != 5:
                    continue

                classe, x, y, w, h = map(float, valores)

                x *= largura
                y *= altura
                w *= largura
                h *= altura

                retangulo = patches.Rectangle(
                    (x - w/2, y - h/2),
                    w, h,
                    linewidth=2,
                    edgecolor="red" if classe == 0 else "blue",
                    facecolor="none"
                )

                ax.add_patch(retangulo)

        ax.set_title(f'{linha["Conjunto"]} - {linha["Ação"]}')
        ax.axis("off")

    plt.suptitle(f"Grupo {grupo} - Comparação das anotações")
    plt.tight_layout()
    plt.show()

In [ ]:
# @title
# Backup do dataset
!cp -r /content/dataset /content/dataset_backup

In [ ]:
# @title
from pathlib import Path

removidos = 0

for _, linha in tabela_plano.iterrows():

    if linha["Ação"] != "REMOVER":
        continue

    caminho_imagem = Path(linha["Caminho"])
    caminho_label = (
        caminho_imagem.parent.parent
        / "labels"
        / f"{caminho_imagem.stem}.txt"
    )

    if caminho_imagem.exists():
        caminho_imagem.unlink()

    if caminho_label.exists():
        caminho_label.unlink()

    removidos += 1

print(f"Imagens removidas: {removidos}")

In [ ]:
# @title
from pathlib import Path

for conjunto in ["train", "valid", "test"]:

    pasta = Path(f"/content/dataset/content/dataset/{conjunto}/images")

    quantidade = len([
        x for x in pasta.iterdir()
        if x.is_file() and x.suffix.lower() in
        [".jpg", ".jpeg", ".png", ".bmp", ".webp"]
    ])

    print(f"{conjunto}: {quantidade} imagens")

# Compactar pasta

In [ ]:
# @title
!zip -r avaliacoes_finais_modelos.zip /content/avaliacoes_finais